# Sandwich Attack Overview Analysis

Comprehensive analysis of all detected sandwich attacks, epoch **946–956**, covering three categories: `standard` / `multi_split` / `diff_signer_owner`.

In [ ]:
# ── Cell 0: Imports & Config ─────────────────────────────────────────────
import sys, os, gc, time
sys.path.insert(0, '../evaluator')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from matplotlib.backends.backend_pdf import PdfPages
import requests
from IPython.display import display
from itables import show

from utils.db import get_client

EPOCHS      = (946, 960)
START_SLOT  = 946 * 432_000   # 408_672_000
END_SLOT    = 961 * 432_000   # 413_424_000
TAG         = '946_960'
DATA_DIR    = '../evaluator/data/3_signer_filter'
PRICES_CSV  = '../evaluator/data/token_prices/prices.csv'
MORALIS_KEY = os.environ['MORALIS_KEY']
MORALIS_BASE = 'https://solana-gateway.moralis.io/token/mainnet'
SOL_ADDR     = 'So11111111111111111111111111111111111111112'
CATEGORIES   = ['standard', 'multi_split', 'diff_signer_owner']

print(f'Config ready — epoch {EPOCHS[0]}-{EPOCHS[1]}, slots {START_SLOT:,}-{END_SLOT:,}')


In [ ]:
# ── Cell 1c: Load all bot_signers ───────────────────────────────────────
dfs = []
for cat in CATEGORIES:
    path = f'{DATA_DIR}/{cat}/bot_signers_{TAG}.parquet'
    df = pd.read_parquet(path)
    df.index.name = 'signer'
    df = df.reset_index()
    df['category'] = cat
    dfs.append(df)
    print(f'  {cat}: {len(df):,} bots')

all_bot = pd.concat(dfs, ignore_index=True)
del dfs
print(f'\nTotal bots: {len(all_bot):,}  |  unique signers: {all_bot["signer"].nunique():,}')
print(f'category dist:\n{all_bot["category"].value_counts().to_string()}')

# ── Duplicate signers across categories ──────────────────────────────────
dup_signers = all_bot[all_bot.duplicated(subset=['signer'], keep=False)]['signer'].unique()
print(f'\nDuplicate signers: {len(dup_signers):,}')

cols = ['signer', 'category', 'sandwich_count', 'active_slot_span',
        'sandwich_freq', 'win_rate', 'in_block_count', 'cross_block_count', 'usd_total_profit']
avail = [c for c in cols if c in all_bot.columns]
dup_df = (
    all_bot[all_bot['signer'].isin(dup_signers)][avail]
    .sort_values(['signer', 'category'])
    .reset_index(drop=True)
)


In [ ]:
# ── Cell 1d: per-category metrics  ──
SEP = '=' * 72
JITO_TIERS = {'jito_and_signal', 'jito_only'}
SOL_ADDR   = 'So11111111111111111111111111111111111111112'

grand = dict(bots=0, sandwiches=0, usd=0.0, sol=0.0, jito_bots=0,
             jito_sw=0, jito_usd=0.0)

for cat in CATEGORIES:
    bot_cat = all_bot[all_bot['category'] == cat]
    sw_cat  = all_sw[all_sw['category'] == cat]

    n_bots      = len(bot_cat)
    n_sw        = len(sw_cat)
    total_usd   = bot_cat['usd_total_profit'].sum()
    total_sol   = sw_cat[sw_cat['token_a'].astype(str) == SOL_ADDR]['profit'].sum()
    avg_usd     = bot_cat['usd_total_profit'].mean()
    med_usd     = bot_cat['usd_total_profit'].median()
    max_usd     = bot_cat['usd_total_profit'].max()
    avg_sw      = bot_cat['sandwich_count'].mean()
    med_sw      = bot_cat['sandwich_count'].median()

    jito_mask   = bot_cat['tier'].isin(JITO_TIERS)
    n_jito      = jito_mask.sum()
    jito_usd    = bot_cat.loc[jito_mask, 'usd_total_profit'].sum()
    jito_sw     = sw_cat['jito_bundle'].fillna(False).sum()

    print(SEP)
    print(f'  [{cat}]')
    print(SEP)
    print(f'  Final bots              : {n_bots:>10,}')
    print(f'  Bot sandwiches          : {n_sw:>10,}')
    print(f'  Total USD profit        : ${total_usd:>12,.2f}')
    print(f'  Total SOL profit        : {total_sol:>12,.2f} SOL')
    print(f'  Avg  USD / bot          : ${avg_usd:>12,.0f}')
    print(f'  Median USD / bot        : ${med_usd:>12,.0f}')
    print(f'  Max  USD / bot          : ${max_usd:>12,.0f}')
    print(f'  Avg  sandwiches / bot   : {avg_sw:>12,.1f}')
    print(f'  Median sandwiches / bot : {med_sw:>12,.1f}')
    print(f'  Jito bots               : {n_jito:>10,}')
    print(f'  Jito-bundle sandwiches  : {int(jito_sw):>10,}')
    print(f'  Jito bot USD profit     : ${jito_usd:>12,.2f}')
    print()

    grand['bots']      += n_bots
    grand['sandwiches']+= n_sw
    grand['usd']       += total_usd
    grand['sol']       += total_sol
    grand['jito_bots'] += n_jito
    grand['jito_sw']   += int(jito_sw)
    grand['jito_usd']  += jito_usd

# De-duplicate: categories may share sandwiches/bots; report grand total from deduped data
all_sw.drop_duplicates(subset=['sandwichId'], inplace=True)
all_bot.drop_duplicates(subset=['signer'], inplace=True)

g_bots     = len(all_bot)
g_sw       = len(all_sw)
g_usd      = all_bot['usd_total_profit'].sum()
g_sol      = all_sw[all_sw['token_a'].astype(str) == SOL_ADDR]['profit'].sum()
g_jbots    = all_bot['tier'].isin(JITO_TIERS).sum()
g_jsw      = int(all_sw['jito_bundle'].fillna(False).sum())
g_jusd     = all_bot.loc[all_bot['tier'].isin(JITO_TIERS), 'usd_total_profit'].sum()
days       = 29.65

print(SEP)
print('  GRAND TOTAL (after dedup)')
print(SEP)
print(f'  Total bots              : {g_bots:>10,}')
print(f'  Total bot sandwiches    : {g_sw:>10,}')
print(f'  Total USD profit        : ${g_usd:>12,.2f}')
print(f'  Total SOL profit        : {g_sol:>12,.2f} SOL')
print(f'  Total Jito bots         : {g_jbots:>10,}')
print(f'  Jito-bundle sandwiches  : {g_jsw:>10,}')
print(f'  Jito bot USD profit     : ${g_jusd:>12,.2f}')
print(f'  Annualised (~{days} days) : ${g_usd / days * 365:>12,.0f} / yr')


In [ ]:
# ── Cell 1b: Token concentration — tokens needed to cover X% of sandwiches ─
token_counts = all_sw['token_a'].value_counts().reset_index()
token_counts.columns = ['token', 'count']
token_counts['cum_pct'] = token_counts['count'].cumsum() / len(all_sw) * 100

THRESHOLDS = [50, 80, 90, 95, 99]
print(f'Distinct tokens: {len(token_counts):,}   Total sandwiches: {len(all_sw):,}\n')
print(f'  {"Coverage":>8}  {"Tokens needed":>14}  {"Actual cum%":>12}')
print(f'  {"-"*38}')
for pct in THRESHOLDS:
    n = int((token_counts['cum_pct'] < pct).sum()) + 1
    actual = token_counts['cum_pct'].iloc[n - 1]
    print(f'  {pct:>7}%  {n:>14,}  {actual:>11.2f}%')

n80 = int((token_counts['cum_pct'] < 80).sum()) + 1
print(f'\n  → {n80} tokens cover ≥80% of all sandwiches')

In [ ]:
# ── Cell 3: Timestamps from DB (2-point interpolation, ~0 DB overhead) ───
client = get_client()

ref_df = client.query_df(
    f'SELECT min(slot) AS slot_min, max(slot) AS slot_max, '
    f'min(timestamp) AS ts_min, max(timestamp) AS ts_max '
    f'FROM sandwiches '
    f'WHERE slot >= {START_SLOT} AND slot < {END_SLOT}'
)

slot_min = int(ref_df['slot_min'].iloc[0])
slot_max = int(ref_df['slot_max'].iloc[0])

def to_utc(val):
    ts = pd.Timestamp(val)
    return ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')

epoch_start_ts = to_utc(ref_df['ts_min'].iloc[0])
epoch_end_ts   = to_utc(ref_df['ts_max'].iloc[0])

print(f'Slot range:  {slot_min:,} – {slot_max:,}')
print(f'UTC range:   {epoch_start_ts.strftime("%Y-%m-%d %H:%M")} – {epoch_end_ts.strftime("%Y-%m-%d %H:%M")}')

# Linear interpolation of timestamps for all sandwiches
ns_min = float(epoch_start_ts.value)
ns_max = float(epoch_end_ts.value)
all_sw['ts'] = pd.to_datetime(
    np.interp(
        all_sw['slot'].values.astype(float),
        [float(slot_min), float(slot_max)],
        [ns_min, ns_max]
    ).astype('int64'),
    utc=True
)
print(f'Timestamps assigned: {all_sw["ts"].notna().sum():,} rows')

In [ ]:
# ── Cell 4: Section 2 — Overview Report ──────────────────────────────────
SEP = '=' * 70

print(SEP)
print('  SECTION 2: OVERVIEW')
print(SEP)

# Time range
duration_days = (epoch_end_ts - epoch_start_ts).total_seconds() / 86400
print(f'\n  Epoch range:               {EPOCHS[0]} – {EPOCHS[1]}')
print(f'  Start (UTC):               {epoch_start_ts.strftime("%Y-%m-%d %H:%M:%S UTC")}')
print(f'  End   (UTC):               {epoch_end_ts.strftime("%Y-%m-%d %H:%M:%S UTC")}')
print(f'  Duration:                  {duration_days:.1f} days')

# Sandwich counts
total_sw = len(all_sw)
print(f'\n  Total sandwiches:          {total_sw:>12,}')
for cat, cnt in all_sw['category'].value_counts().items():
    print(f'    {cat:<26} {int(cnt):>10,}')

total_sw = all_bot['sandwich_count'].sum()
print(f'\n  Total sandwiches:          {total_sw:>12,}')
for cat, cnt in all_sw['category'].value_counts().items():
    print(f'    {cat:<26} {int(cnt):>10,}')

# USD profit
total_usd   = all_sw['usd_profit'].sum()
total_sol   = all_sw[all_sw['token_a'] == 'SOL']['profit'].sum()
print(f'\n  Total USD profit:          ${total_usd:>13,.2f}')
print(f'  Total SOL profit:           {total_sol:>13,.4f} SOL')

# Victim transactions
total_victim_txs = int(all_sw['victim_count'].sum())
print(f'\n  Total victim transactions: {total_victim_txs:>12,}')

# Distinct victim signers (DB)
print('\n  Querying distinct victim signers (DB)...')
vs_df = client.query_df(
    'SELECT count(DISTINCT vs) AS n FROM ('
    '  SELECT arrayJoin(st.signers) AS vs'
    '  FROM sandwich_txs st'
    '  INNER JOIN sandwiches s ON st.sandwichId = s.sandwichId'
    f' WHERE s.slot >= {START_SLOT} AND s.slot < {END_SLOT}'
    '    AND (s.signerSame = true'
    '         OR (s.signerSame = false AND s.ownerSame = true'
    '             AND s.multiFrontRun = false AND s.multiBackRun = false))'
    "    AND st.type = 'victim'"
    ')'
)
n_distinct_vs = int(vs_df['n'].iloc[0])
print(f'  Distinct victim signers:   {n_distinct_vs:>12,}')

# Sandwich with most victim txs
max_vic_idx = all_sw['victim_count'].idxmax()
max_vic     = all_sw.loc[max_vic_idx]
print(f'\n  Sandwich with most victim txs:')
print(f'    sandwichId:    {max_vic["sandwichId"]}')
print(f'    victim_count:  {int(max_vic["victim_count"])}')
print(f'    slot:          {int(max_vic["slot"]):,}')
print(f'    token_a:       {max_vic["token_a"]}')
print(f'    profit:        {max_vic["profit"]:.6f}')
print(f'    category:      {max_vic["category"]}')

In [ ]:
# ── Cell 5: Section 3 — Top-20 TokenA ──────────────────────────────────────
prices_df    = pd.read_csv(PRICES_CSV)  # symbol/name lookup from local CSV
top20_counts = all_sw['token_a'].value_counts().head(20)
top20_tokens = top20_counts.index.tolist()
total_sw     = len(all_sw)

print('=' * 70)
print('  SECTION 3: TOP-20 tokenA')
print('=' * 70)

sym_map  = dict(zip(prices_df['token'], prices_df['symbol']))
name_map = dict(zip(prices_df['token'], prices_df['name']))

rows = []
for i, token in enumerate(top20_tokens):
    cnt = int(top20_counts.iloc[i])
    rows.append({
        'rank':            i + 1,
        'symbol':          sym_map.get(token, ''),
        'token':           token,
        'sandwich_count':  cnt,
        'pct':             cnt / total_sw * 100,
    })

top20_df = pd.DataFrame(rows).set_index('rank')
print(f'\n  {"#":>2}  {"Symbol":>8}  {"Token (first 16)":>18}  {"Count":>10}  {"% of total":>10}')
print(f'  {"-"*56}')
for idx, row in top20_df.iterrows():
    tok_short = (row['token'][:16] + '...') if len(row['token']) > 16 else row['token']
    print(f'  {idx:>2}  {row["symbol"]:>8}  {tok_short:>18}  {int(row["sandwich_count"]):>10,}  {row["pct"]:>9.2f}%')

display(top20_df)

In [ ]:
# ── Cell 7: Daily user transaction counts from DB (aggregated server-side) ─
# Interpolation params from Cell 3
ts_min_sec  = int(epoch_start_ts.timestamp())
slope_sec   = (epoch_end_ts.timestamp() - epoch_start_ts.timestamp()) / max(slot_max - slot_min, 1)

print('Querying daily user transaction counts from ClickHouse...')
daily_txs = client.query_df(
    f'SELECT '
    f'  toDate(fromUnixTimestamp(toUInt32('
    f'    {ts_min_sec} + {slope_sec:.8f} * (toInt64(slot) - {slot_min})'
    f'  ))) AS date, '
    f'  sum(txCount) AS total_txs '
    f'FROM slot_txs '
    f'WHERE slot >= {START_SLOT} AND slot < {END_SLOT} '
    f'GROUP BY date ORDER BY date'
)
daily_txs['date'] = pd.to_datetime(daily_txs['date'], utc=True)
print(f'  {len(daily_txs)} daily rows')
print(daily_txs.to_string(index=False))

In [ ]:
# ── Cell 8: DEX-breakdown — derive raw_dex from sandwich_txs.programs ───

client = get_client()

# ── 1. Two separate program maps (derived from watcher/programs.yaml) ─────
# DEX/AMM programs
PROGRAM_DEX = {
    '6EF8rrecthR5Dkzon8Nwu78hRvfCKubJ14M5uBEwF6P': 'pumpfun',
    'pAMMBay6oceH9fJKBRHGP5D4bD4sWpmSwMn52FMfXEA': 'pumpfun_amm',
    '675kPX9MHTjS2zt1qfr1NYHuzeLXfQM9H24wFSUt1Mp8': 'raydium_v4',
    'routeUGWgWzqBWFcrCfv8tritsqukccJPu3q5GPP3xS':  'raydium_v4',
    'CPMMoo8L3F4NbTegBCKVNunggL7H1ZpdTHKxQB5qKP1C': 'raydium_cpmm',
    'CAMMCzo5YL8w4VFF8KVHrK22GGUsp5VTaW7grrKgrWqK': 'raydium_clmm',
    'Eo7WjKq67rjJQSZxS6z3YkapzY3eMj6Xy8X5EQVn5UaB': 'meteora_damm',
    'LBUZKhRxPF3XUpBCjp4YzTKgLccjZhTSDM9YuVaPwxo':  'meteora_dlmm',
    'dbcij3LWUppWqq96dh6gJWwBifmcGfLSB5D4DuSMaqN':  'meteora_dbc',
    'cpamdpZCGKUy5JxQXB4dcpGPiikHawvSWAd6mEn1sGG':  'meteora_damm_v2',
    '9W959DqEETiGZocYWCQPaJ6sBmUzgfxXfqGeTEdp3aQP': 'orca_token_swap',
    'DjVE6JNiYqPL2QXyCUUh8rNjHrbz9hXHNYt99MQ59qw1': 'orca_token_swap',
    'whirLbMiicVdio4qvUfM5KAg6Ct8VwpYzGff3uctyCc':  'whirlpool',
    'HpNfyc2Saw7RKkQd8nEL4khUcuPhQ7WwY1B2qjx8jxFq': 'pancakeswap',
    'Dooar9JkhdZ7J3LHN3A7YCuoGRUggXhQaG4kijfLGU2j': 'stepn_dooar',
    'REALQqNEomY6cQGZJUGwywTBD2UmDT32rZcNnfxQ5N2':  'byreal_clmm',
    # PropAMM
    'SV2EYYJyRz2YhfXwXnhNAevDEui5Q6yrfyo13WtupPF':  'solfi_v2',
    'BiSoNHVpsVZW2F7rx2eQ59yQwKxzU5NvBcmKshCSUypi': 'bisonfi',
    '9H6tua7jkLhdm3w8BvgpTn5LZNU7g4ZynDmCiNN3q6Rp': 'humidifi',
    'fUSioN9YKKSa3CUC2YUc4tPkHJ5Y6XW1yz8y6F7qWz9':  'fusion_amm',
    'TessVdML9pBGgG9yGks7o4HewRaXVAMuoVj4x83GLQH':  'tessera_v',
    'goonuddtQRrWqqn5nFyczVKaie28f3kDkHWkHtURSLE':   'goonfi_v2',
    'ALPHAQmeA7bjrVuccPsYPiCvsi428SNwte66Srvs4pHA':  'alphaq',
    'obriQD1zbpyLz95G5n7nJe6a4DPjpFwa5XYPoNm113y':  'obric_v2',
    'ZERor4xhbUycZ6gb9ntrhqscUcZmAbQDjEAtCf4hbZY':  'zerofi',
}

# Aggregator programs — routers that sit on top of DEXes
PROGRAM_AGG = {
    '6m2CDdhRgxpH4WjvdzxAYbGxwdGUz5MziiL5jek2kBma': 'okx_agg',
    'JUP6LkbZbjS1jKKwapdHNy74zcZ3tLUZoi5QNyVTaV4':  'jupiter_agg',
    'DCA265Vj8a9CEuX1eb1LWRnDT7uK6q1xMipnNyatn23M':  'jupiter_dca',
    'DF1ow4tspfHX9JwWJsAb9epbkA8hmpSEAtxXy1V27QBH':  'dflow_agg',
    'AxiomfHaWDemCFBLBayqnEnNwE6b7B2Qz3UmzMpgbMG6':  'axiom_agg',
}

PROGRAM_RAW_DEX = {**PROGRAM_DEX, **PROGRAM_AGG}  # for debug helper
print(f'PROGRAM_DEX: {len(PROGRAM_DEX)} entries   PROGRAM_AGG: {len(PROGRAM_AGG)} entries')

# ── 2. Query: collect all programs per sandwich ───────────────────────────
prog_q = (
    'SELECT sandwichId, groupUniqArray(p) AS prog_list '
    'FROM sandwich_txs '
    'ARRAY JOIN programs AS p '
    f'WHERE slot >= {START_SLOT} AND slot < {END_SLOT} '
    'GROUP BY sandwichId'
)
prog_df = client.query_df(prog_q)
print(f'Sandwiches with program data: {len(prog_df):,}')

# ── 3. Map programs → DEX set & aggregator set per sandwich ──────────────
prog_df['dex_set'] = prog_df['prog_list'].apply(
    lambda pl: {PROGRAM_DEX[p] for p in pl if p in PROGRAM_DEX}
)
prog_df['agg_set'] = prog_df['prog_list'].apply(
    lambda pl: {PROGRAM_AGG[p] for p in pl if p in PROGRAM_AGG}
)
prog_df['n_dex'] = prog_df['dex_set'].apply(len)

def _resolve(row):
    ds, ag = row['dex_set'], row['agg_set']
    if len(ds) == 1:
        return next(iter(ds))
    if len(ds) == 0:
        return next(iter(ag)) if ag else 'unknown'
    return '__multi__'

prog_df['raw_dex'] = prog_df.apply(_resolve, axis=1)

# ── 4. Resolve known multi-DEX combinations ───────────────────────────────
multi = prog_df[prog_df['raw_dex'] == '__multi__']
print(f'\n=== Sandwiches with multiple DEX programs: {len(multi):,} '
      f'({len(multi)/len(prog_df)*100:.3f}%) ===')
if len(multi) > 0:
    breakdown = multi['dex_set'].apply(frozenset).value_counts().head(20)
    print(breakdown.to_string())
    def _resolve_multi(s):
        fs = frozenset(s)
        if fs == frozenset({'pumpfun', 'pumpfun_amm'}):
            return 'pumpfun'
        if fs == frozenset({'raydium_v4', 'raydium_clmm'}):
            return 'raydium_v4'
        return 'unknown'
    prog_df.loc[prog_df['raw_dex'] == '__multi__', 'raw_dex'] = (
        prog_df.loc[prog_df['raw_dex'] == '__multi__', 'dex_set'].apply(_resolve_multi)
    )
else:
    print('No exceptions — every sandwich maps to at most one DEX.')

# ── 5. Merge raw_dex into all_sw ──────────────────────────────────────────
all_sw = all_sw.drop(columns=['raw_dex'], errors='ignore')
all_sw = all_sw.merge(prog_df[['sandwichId', 'raw_dex']], on='sandwichId', how='left')
all_sw['raw_dex'] = all_sw['raw_dex'].fillna('unknown')
print()
display(all_sw['raw_dex'].value_counts())

In [ ]:
# ── 2. DEX name mapping dict (manually specified) ──────────────────────
# value = display name in charts; entries mapped to 'Other DEXes' are merged
DEX_DISPLAY = {
    '':                'Other DEXes',
    'pumpfun':         'Pump.fun',
    'pumpfun_amm':     'Pump.fun',
    'raydium_v4':      'Raydium',
    'raydium_cpmm':    'Raydium',
    'raydium_clmm':    'Raydium',
    'meteora_damm_v2': 'Meteora',
    'meteora_dlmm':    'Meteora',
    'meteora_dbc':     'Meteora',
    'whirlpool':       'Whirlpool',
    'orca_token_swap': 'Other DEXes',
    'pancakeswap':     'Pancakeswap',
    # 'jupiter_agg':      'Jupiter',
    # 'dflow_agg':        'DFlow'
}
# DEX display order, Other DEXes last
DEX_ORDER = [
    'Meteora',
    'Pump.fun', 
    'Raydium', 
    'Pancakeswap',
    'Whirlpool',
    'Other DEXes',
]
DEX_COLORS = {
    'Meteora':         '#A82203',
    'Pump.fun':        '#208CC0',
    # 'Pump.fun AMM':    '#F4A261',A64036
    'Raydium':      '#F1AF3A',
    'Pancakeswap':  '#CF5E4E',
    'Whirlpool':    '#637B31',
    # 'Raydium CPMM':    '#6BAED6',
    # 'Raydium CLMM':    '#9ECAE1',
    'Other DEXes':     '#003967',
}
# DEX_COLORS = {
#     'Meteora':         '#A82203',
#     'Pump.fun':        '#208CC0',
#     # 'Pump.fun AMM':    '#F4A261',A64036
#     'Raydium':      '#F1AF3A',
#     'Pancakeswap':  '#CF5E4E',
#     'Whirlpool':    '#637B31',
#     'Jupiter':      '#B47880',
#     'DFlow':        '#635761',
#     # 'Raydium CPMM':    '#6BAED6',
#     # 'Raydium CLMM':    '#9ECAE1',
#     'Other DEXes':     '#003967',
# }


In [ ]:
# ── 3. DEX share stats after mapping ──────────────────────────────────────
all_sw['dex'] = all_sw['raw_dex'].map(DEX_DISPLAY).fillna('Other DEXes')
display(all_sw['dex'].value_counts())

# volume & profit proportion per DEX
dex_summary = (
    all_sw.groupby('dex', observed=True)
    .agg(count=('sandwichId', 'count'), total_profit=('usd_profit', 'sum'))
    .reindex(DEX_ORDER)
)
dex_summary = dex_summary.fillna(0)
dex_summary['vol_pct%']    = (dex_summary['count']        / dex_summary['count'].sum()        * 100).round(2)
dex_summary['profit_pct%'] = (dex_summary['total_profit'] / dex_summary['total_profit'].sum() * 100).round(2)
dex_summary['avg_profit']  = (dex_summary['total_profit'] / dex_summary['count']).round(4)
dex_summary.index.name = 'DEX'
display(dex_summary[['count', 'vol_pct%', 'total_profit', 'profit_pct%', 'avg_profit']])

SEP = '=' * 68
print(SEP)
print('  DEX BREAKDOWN — Volume & Profit')
print(SEP)
print(f'\n  {"DEX":<14} {"Count":>10} {"Vol%":>7} {"Total USD":>14} {"Prof%":>7} {"Avg USD":>10}')
print(f'  {"-"*66}')
for dex, r in dex_summary.iterrows():
    print(f'  {str(dex):<14} {int(r["count"]):>10,} {r["vol_pct%"]:>6.2f}%'
          f'  ${r["total_profit"]:>12,.2f}  {r["profit_pct%"]:>6.2f}%  ${r["avg_profit"]:>8,.4f}')

# ── 4. Aggregate by date × DEX → pivot ────────────────────────────────────
daily_sw_dex = (
    all_sw
    .groupby([all_sw['ts'].dt.floor('D'), 'dex'])
    .agg(usd_profit=('usd_profit', 'sum'), count=('sandwichId', 'count'))
    .reset_index()
    .rename(columns={'ts': 'date'})
)

# keep daily_sw for downstream cells
daily_sw = daily_sw_dex.groupby('date')[['usd_profit', 'count']].sum().reset_index()

active_dexes = [d for d in DEX_ORDER if d in daily_sw_dex['dex'].unique()]
vol_pivot  = daily_sw_dex.pivot_table(index='date', columns='dex', values='count',
                                       aggfunc='sum', fill_value=0)[active_dexes]
prof_pivot = daily_sw_dex.pivot_table(index='date', columns='dex', values='usd_profit',
                                       aggfunc='sum', fill_value=0)[active_dexes]


In [ ]:
# ── Cell Cost-A: Query fr/br fees + bundle tips, save parquet ────────────

import os

COST_PARQUET = f'../evaluator/data/sandwich_costs_{TAG}.parquet'

# ── 1. fr/br fees per sandwich ────────────────────────────────────────────
print('Querying fr/br fees...')
fee_df = client.query_df(f"""
SELECT st.sandwichId, sum(st.fee) AS fee_lamports
FROM sandwich_txs st
INNER JOIN sandwiches s ON st.sandwichId = s.sandwichId
WHERE s.slot >= {START_SLOT} AND s.slot < {END_SLOT}
  AND st.type IN ('frontRun', 'backRun')
GROUP BY st.sandwichId
""")
print(f'  {len(fee_df):,} sandwiches with fee data')

# ── 2. Bundle tips — only for jito_bundle sandwiches ─────────────────────
bundle_sids = all_sw.loc[all_sw['jito_bundle'].fillna(False), 'sandwichId'].tolist()
print(f'\nBundle sandwiches: {len(bundle_sids):,}')

tip_df = None
if bundle_sids:
    # Get fr/br signatures for bundle sandwiches
    sid_str  = "','".join(bundle_sids)
    print('  Querying fr/br signatures for bundle sandwiches...')
    sig_df = client.query_df(f"""
    SELECT st.sandwichId, st.signature, st.slot
    FROM sandwich_txs st
    INNER JOIN sandwiches s ON st.sandwichId = s.sandwichId
    WHERE s.slot >= {START_SLOT} AND s.slot < {END_SLOT}
      AND st.type IN ('frontRun', 'backRun')
      AND st.inBundle = 1
      AND st.sandwichId IN ('{sid_str}')
    """)
    print(f'  {len(sig_df):,} inBundle fr/br txs')

    # For each slot, look up bundleId + tip from jito_bundles
    slots_str = ','.join(map(str, sig_df['slot'].unique()))
    print('  Querying jito_bundles for tips...')
    jb_df = client.query_df(f"""
    SELECT bundleId, arrayJoin(transactions) AS sig, landedTipLamports AS tip_lamports
    FROM jito_bundles
    WHERE slot IN ({slots_str})
    """)
    print(f'  {len(jb_df):,} jito_bundles txs')

    # Join: sandwich signature → bundleId + tip
    sig_with_tip = sig_df.merge(jb_df, left_on='signature', right_on='sig', how='left')

    # Per sandwich: sum of unique bundle tips (a sandwich may span 1-2 bundles)
    tip_df = (
        sig_with_tip.dropna(subset=['bundleId'])
        .drop_duplicates(subset=['sandwichId', 'bundleId'])
        .groupby('sandwichId', as_index=False)['tip_lamports'].sum()
    )
    print(f'  {len(tip_df):,} sandwiches with tip data')

# ── 3. Merge fees + tips, convert to USD ─────────────────────────────────
prices_df = pd.read_csv(PRICES_CSV)
sol_row    = prices_df[prices_df['token'] == SOL_ADDR]
sol_price  = float(sol_row['usd_price'].iloc[0]) if len(sol_row) else 0.0
print(f'\nSOL price: ${sol_price:.4f}')

cost_df = fee_df.copy()
cost_df['fee_lamports'] = cost_df['fee_lamports'].astype(float)

if tip_df is not None and len(tip_df) > 0:
    cost_df = cost_df.merge(tip_df, on='sandwichId', how='left')
else:
    cost_df['tip_lamports'] = 0.0

cost_df['tip_lamports'] = cost_df['tip_lamports'].fillna(0).astype(float)
cost_df['fee_usd']  = cost_df['fee_lamports']  / 1e9 
cost_df['tip_usd']  = cost_df['tip_lamports']  / 1e9 
cost_df['cost_usd'] = cost_df['fee_usd'] + cost_df['tip_usd']

print(f'\nCost summary (USD):')
print(f'  fee_usd  — mean: ${cost_df["fee_usd"].mean():.4f}  total: ${cost_df["fee_usd"].sum():,.2f}')
print(f'  tip_usd  — mean: ${cost_df["tip_usd"].mean():.4f}  total: ${cost_df["tip_usd"].sum():,.2f}')
print(f'  cost_usd — mean: ${cost_df["cost_usd"].mean():.4f}  total: ${cost_df["cost_usd"].sum():,.2f}')

# ── 4. Save parquet ───────────────────────────────────────────────────────
cost_df.to_parquet(COST_PARQUET, index=False)
print(f'\nSaved: {COST_PARQUET}  ({len(cost_df):,} rows)')


In [ ]:
# ── Cell Cost-B: Volume+UserTx / Profit chart ────────────────────────────

# Load cost parquet (idempotent re-run)
if 'cost_df' not in dir() or cost_df is None:
    cost_df = pd.read_parquet(COST_PARQUET)

# Ensure sw_class exists (defined in Cell 9a; derive here if not yet run)
if 'sw_class' not in all_sw.columns:
    _cond = [all_sw['consecutive'], ~all_sw['consecutive'] & all_sw['cross_block']]
    all_sw['sw_class'] = np.select(_cond, ['inblock', 'cross_block'], default='inblock')

# Daily aggregation — split by sw_class
_sw_c = all_sw.merge(cost_df[['sandwichId', 'fee_lamports', 'tip_lamports']], on='sandwichId', how='left')
_sw_c['cost_lamports'] = _sw_c['fee_lamports'].fillna(0) + _sw_c['tip_lamports'].fillna(0)

daily_by_class = (
    _sw_c.groupby([_sw_c['ts'].dt.floor('D'), 'sw_class'])
    .agg(count=('sandwichId', 'count'), profit=('usd_profit', 'sum'))
    .reset_index()
    .rename(columns={'ts': 'date'})
)
daily_by_class['date'] = pd.to_datetime(daily_by_class['date'], utc=True)

# Pivot to wide
count_pivot  = daily_by_class.pivot(index='date', columns='sw_class', values='count').fillna(0)
profit_pivot = daily_by_class.pivot(index='date', columns='sw_class', values='profit').fillna(0)

# Remove last day (partial)
count_pivot  = count_pivot.iloc[:-1]
profit_pivot = profit_pivot.iloc[:-1]

txs_map = daily_txs.set_index('date')['total_txs']
_txs = txs_map[txs_map.index < count_pivot.index[-1] + pd.Timedelta(days=1)]
# Also drop last day from txs
_txs = _txs.iloc[:-1] if len(_txs) > len(count_pivot) else _txs

# ── Color palette (scheme B: blue for count / orange for profit) ──────────
C_IB_V   = '#92C5DE'   # count  — in-block   (light blue)
C_CB_V   = '#2166AC'   # count  — cross-block (dark blue)
C_IB_P   = '#F4A582'   # profit — in-block   (light orange)
C_CB_P   = '#D6604D'   # profit — cross-block (dark brick red)
C_LINE   = '#737373'   # user tx line (mid grey)

bw_total = pd.Timedelta(hours=14)

# Ensure columns exist
for col in ['inblock', 'cross_block']:
    if col not in count_pivot.columns:
        count_pivot[col]  = 0
        profit_pivot[col] = 0

# ── Figure ────────────────────────────────────────────────────────────────
fig2, (ax_v, ax_p) = plt.subplots(2, 1, figsize=(16, 13), sharex=True)
fig2.subplots_adjust(bottom=0.15, hspace=0.08)

dates = count_pivot.index

# Panel (a): stacked bar (in-block + cross-block) + User Tx line
ax_v.bar(dates, count_pivot['inblock'],    width=bw_total, color=C_IB_V, alpha=0.88, label='In-block')
ax_v.bar(dates, count_pivot['cross_block'], width=bw_total, color=C_CB_V, alpha=0.88,
         bottom=count_pivot['inblock'], label='Cross-block')
ax_v.set_ylabel('Sandwich Count', fontsize=28)
ax_v.yaxis.set_major_formatter(make_sci_fmt())
ax_v.yaxis.get_offset_text().set_fontsize(24)
ax_v.tick_params(axis='x', which='both', length=0, labelbottom=False, labelsize=20)
ax_v.tick_params(axis='y', labelsize=24)
ax_v.spines['top'].set_visible(False)
add_grid(ax_v)

ax_vr = ax_v.twinx()
ax_vr.plot(_txs.index, _txs.values, color=C_LINE, marker='s', markersize=4,
           linewidth=1.5, linestyle='--', label='User Tx Volume')
ax_vr.set_ylabel('Daily User Transactions', fontsize=28)
ax_vr.yaxis.set_major_formatter(make_sci_fmt())
ax_vr.yaxis.get_offset_text().set_fontsize(24)
ax_vr.tick_params(axis='y', labelsize=24)
ax_vr.spines['top'].set_visible(False)

total_count = count_pivot['inblock'] + count_pivot['cross_block']
ax_v.set_ylim(0, total_count.max() * 1.35)
ax_vr.set_ylim(1.5e8, _txs.values.max() * 1.15)

h1, l1 = ax_v.get_legend_handles_labels()
h2, l2 = ax_vr.get_legend_handles_labels()
ax_v.legend(h1 + h2, l1 + l2, loc='upper right', ncol=2, fontsize=23, framealpha=0.85)

# Panel (b): stacked profit bar (in-block + cross-block)
ax_p.bar(dates, profit_pivot['inblock'],    width=bw_total, color=C_IB_P, alpha=0.88, label='In-block')
ax_p.bar(dates, profit_pivot['cross_block'], width=bw_total, color=C_CB_P, alpha=0.88,
         bottom=profit_pivot['inblock'], label='Cross-block')
ax_p.set_ylabel('Profit (USD)', fontsize=28)
ax_p.yaxis.set_major_formatter(make_sci_fmt())
ax_p.yaxis.get_offset_text().set_fontsize(24)
ax_p.xaxis.set_major_locator(mdates.DayLocator(interval=3, tz='UTC'))
ax_p.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax_p.tick_params(axis='x', labelsize=24, rotation=30)
ax_p.tick_params(axis='y', labelsize=24)
ax_p.spines['top'].set_visible(False)
total_profit = profit_pivot['inblock'] + profit_pivot['cross_block']
ax_p.set_ylim(0, total_profit.max() * 1.1)
ax_p.legend(fontsize=25, framealpha=0.9)
add_grid(ax_p)

pdf_path2 = 'sandwich_cost_chart.pdf'
with PdfPages(pdf_path2) as pdf:
    pdf.savefig(fig2, bbox_inches='tight')
plt.show()
print(f'\nSaved: {pdf_path2}')


In [ ]:
# ── Cell: Profit-range breakdown — attack count & total profit ───────────
import matplotlib.ticker as mticker

# ── Bin definition (USD profit) ───────────────────────────────────────────
PROFIT_BINS   = [-np.inf, -100, -10, -1, 0, 1, 10, 100, 1000, np.inf]
PROFIT_LABELS = ['<-100', '-100--10', '-10-1', '-1-0', '0–1', '1–10', '10–100', '100–1k', '>1k']
# Tick labels at each bin edge (n+1 positions)
EDGE_LABELS   = ['', '-100', '-10', '-1', '0', '1', '10', '100', '1k', '']

all_sw['profit_bin'] = pd.cut(
    all_sw['usd_profit'], bins=PROFIT_BINS, labels=PROFIT_LABELS, right=True
)
bin_stats = (
    all_sw.groupby('profit_bin', observed=True)
    .agg(count=('sandwichId', 'count'),
         total_profit=('usd_profit', 'sum'))
    .reset_index()
)

# ── Plot ──────────────────────────────────────────────────────────────────
BAR_COLOR    = '#5B8DB8'
PROFIT_COLOR = '#C0392B'
FS           = 20   # base font size

n            = len(bin_stats)
bar_centers  = np.arange(n) + 0.5   # bars centered at 0.5, 1.5, …
tick_pos     = np.arange(n + 1)     # ticks at 0, 1, 2, … (bin edges)

fig, ax1 = plt.subplots(figsize=(8, 4.5))

# ── Bars (width=1 → touching, no gap) ────────────────────────────────────
bars = ax1.bar(bar_centers, bin_stats['count'], width=0.85,
               color=BAR_COLOR, alpha=0.85, zorder=3, label='Sandwich Count')

ax1.set_xlim(-0.25, n + 0.25)  # small left/right margin
ax1.set_xticks(tick_pos)
ax1.set_xticklabels(EDGE_LABELS, fontsize=FS - 1)
ax1.set_xlabel('Sandwich Profit (USD)', fontsize=FS)
ax1.set_ylabel('Snadwich Count', fontsize=FS)
ax1.tick_params(axis='y', labelsize=FS - 1, colors='black')
ax1.tick_params(axis='x', labelsize=FS - 1, colors='black')
ax1.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
ax1.yaxis.get_major_formatter().set_scientific(True)
ax1.yaxis.get_major_formatter().set_powerlimits((0, 3))
ax1.set_axisbelow(True)
ax1.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.4, color='#aaaaaa')
for spine in ax1.spines.values():
    spine.set_edgecolor('black')
# Bar count annotations
raw_ymax = bin_stats['count'].max()
ax1.set_ylim(0, raw_ymax * 1.18)  # extra headroom for annotations
ymax = ax1.get_ylim()[1]
# for bar, cnt in zip(bars, bin_stats['count']):
#     ax1.text(bar.get_x() + bar.get_width() / 2,
#              bar.get_height() + ymax * 0.012,
#              f'{cnt:,}', ha='center', va='bottom',
#              fontsize=FS - 3, color='#222222')

# ── Right axis: total profit ──────────────────────────────────────────────
ax2 = ax1.twinx()
ax2.plot(bar_centers, bin_stats['total_profit'],
         color=PROFIT_COLOR, marker='o',
         linewidth=1.8, markersize=6, zorder=4, label='Total Profit (USD)')
ax2.set_ylabel('Total Profit (USD)', fontsize=FS)
ax2.tick_params(axis='y', labelsize=FS - 1, colors='black')
ax2.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
ax2.yaxis.get_major_formatter().set_scientific(True)
ax2.yaxis.get_major_formatter().set_powerlimits((0, 3))
for spine in ax2.spines.values():
    spine.set_edgecolor('black')

ax1.spines['top'].set_visible(False)
ax2.spines['top'].set_visible(False)


# ── Align zeros: push ax1 bottom down to match ax2's zero fraction ───────
y1min, y1max = ax1.get_ylim()
y2min, y2max = ax2.get_ylim()
# fraction from bottom where zero sits on right axis
frac2 = -y2min / (y2max - y2min)   # >0 if y2min<0
# Set ax1 bottom so its zero is at the same fraction:
# -new_y1min / (y1max - new_y1min) = frac2  →  new_y1min = frac2*y1max/(frac2-1)
if frac2 > 0:
    new_y1min = frac2 * y1max / (frac2 - 1)
    ax1.set_ylim(new_y1min, y1max)   # extra bottom whitespace
    ax2.set_ylim(y2min, y2max)

# ── Legend ────────────────────────────────────────────────────────────────
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, fontsize=FS - 2,
           loc='upper left', framealpha=0.9)

fig.tight_layout()
pdf_path = 'sandwich_profit_range.pdf'
with PdfPages(pdf_path) as pdf:
    pdf.savefig(fig, bbox_inches='tight')
plt.show()
print(f'Saved: {pdf_path}')
display(bin_stats.assign(
    total_profit=bin_stats['total_profit'].map('${:,.1f}'.format)
).rename(columns={'profit_bin': 'Profit range', 'count': 'Count', 'total_profit': 'Total profit'}))


In [ ]:
# ── Cell 9a: Classification + Distance ───────────────────────────────────

# 1. Classify from existing parquet columns (no DB needed)
all_sw['sw_class'] = np.where(
    all_sw['cross_block'] | all_sw['consecutive'], 'cross_block', 'inblock'
).astype(str)
all_sw['sw_class'] = all_sw['sw_class'].astype('category')
print('Class distribution:')
print(all_sw['sw_class'].value_counts().to_string())

# 2. Query DB: last-front and first-back (slot, position) per sandwich
print('\nQuerying last-front / first-back positions from DB...')
dist_df = client.query_df(f"""
SELECT
    st.sandwichId,
    argMaxIf(toUInt64(st.slot),    toUInt64(st.slot)*100000 + toUInt64(st.position), st.type = 'frontRun') AS lf_slot,
    argMaxIf(toInt32(st.position), toUInt64(st.slot)*100000 + toUInt64(st.position), st.type = 'frontRun') AS lf_pos,
    argMinIf(toUInt64(st.slot),    toUInt64(st.slot)*100000 + toUInt64(st.position), st.type = 'backRun')  AS fb_slot,
    argMinIf(toInt32(st.position), toUInt64(st.slot)*100000 + toUInt64(st.position), st.type = 'backRun')  AS fb_pos
FROM sandwich_txs st
INNER JOIN sandwiches s ON st.sandwichId = s.sandwichId
WHERE s.slot >= {START_SLOT} AND s.slot < {END_SLOT}
  AND st.type IN ('frontRun', 'backRun')
GROUP BY st.sandwichId
""")
print(f'  {len(dist_df):,} sandwiches with position data')
for col in ['lf_slot', 'fb_slot']:
    dist_df[col] = dist_df[col].astype(int)
for col in ['lf_pos', 'fb_pos']:
    dist_df[col] = dist_df[col].astype(int)

# 3. Same-block distance (vectorized): first_back_pos - last_front_pos - 1
same_mask = dist_df['lf_slot'] == dist_df['fb_slot']
dist_df['distance'] = np.where(
    same_mask,
    dist_df['fb_pos'] - dist_df['lf_pos'] - 1,
    np.nan
)

# 4. Cross-block distance via slot_txs
cross = dist_df[~same_mask]
print(f'  Cross-block sandwiches: {len(cross):,}')

if len(cross) > 0:
    min_s, max_s = int(cross['lf_slot'].min()), int(cross['fb_slot'].max())
    stx = client.query_df(
        f'SELECT slot, txCount FROM slot_txs WHERE slot >= {min_s} AND slot <= {max_s}'
    )
    stx_map = dict(zip(stx['slot'].astype(int), stx['txCount'].astype(int)))
    slot_series = pd.Series(stx_map, dtype=int).sort_index()

    def _cross_dist(row):
        lf_s, lf_p = int(row['lf_slot']), int(row['lf_pos'])
        fb_s, fb_p = int(row['fb_slot']), int(row['fb_pos'])
        d = stx_map.get(lf_s, 0) - lf_p - 1          # txs remaining in lf_slot after last_front
        if fb_s > lf_s + 1:
            d += int(slot_series.loc[lf_s + 1: fb_s - 1].sum())  # full intermediate slots
        d += fb_p                                       # txs before first_back in fb_slot
        return max(d, 0)

    cross_dists = cross.apply(_cross_dist, axis=1)
    dist_df.loc[~same_mask, 'distance'] = cross_dists.values

dist_df['distance'] = dist_df['distance'].fillna(0).clip(lower=0)

# 5. Merge back to all_sw
if 'distance' in all_sw.columns:
    all_sw = all_sw.drop(columns=['distance'])
all_sw = all_sw.merge(dist_df[['sandwichId', 'distance']], on='sandwichId', how='left')
print(f'\nDistance coverage: {all_sw["distance"].notna().sum():,} / {len(all_sw):,}')
print(f'Distance — min:{all_sw["distance"].min():.0f}  '
      f'mean:{all_sw["distance"].mean():.1f}  '
      f'median:{all_sw["distance"].median():.0f}  '
      f'max:{all_sw["distance"].max():.0f}')

In [ ]:
# ── Cell 9b: Statistics table ─────────────────────────────────────────────
SEP = '=' * 72
print(SEP)
print('  SANDWICH CLASSIFICATION & DISTANCE')
print(SEP)

stats = (
    all_sw.groupby('sw_class', observed=True)
    .agg(
        count      = ('sandwichId', 'count'),
        total_usd  = ('usd_profit', 'sum'),
        avg_usd    = ('usd_profit', 'mean'),
        median_usd = ('usd_profit', 'median'),
        min_dist   = ('distance',   'min'),
        max_dist   = ('distance',   'max'),
        median_dist= ('distance',   'median'),
        avg_dist   = ('distance',   'mean'),
    )
    .reset_index()
)
stats['pct']      = stats['count'] / len(all_sw) * 100
stats['win_rate'] = (
    all_sw[all_sw['usd_profit'] > 0]
    .groupby('sw_class', observed=True)['sandwichId'].count()
    .reindex(stats['sw_class'].values)
    .values / stats['count'] * 100
)

total_row = pd.DataFrame([{
    'sw_class':   'ALL',
    'count':      len(all_sw),
    'pct':        100.0,
    'total_usd':  all_sw['usd_profit'].sum(),
    'avg_usd':    all_sw['usd_profit'].mean(),
    'median_usd': all_sw['usd_profit'].median(),
    'min_dist':   all_sw['distance'].min(),
    'max_dist':   all_sw['distance'].max(),
    'median_dist':all_sw['distance'].median(),
    'avg_dist':   all_sw['distance'].mean(),
    'win_rate':   (all_sw['usd_profit'] > 0).mean() * 100,
}])
stats_display = pd.concat([stats, total_row], ignore_index=True)

print(f'\n  {"Class":<14} {"Count":>10} {"Pct%":>7} '
      f'{"Total USD":>14} {"Avg USD":>10} {"Median USD":>12} {"Win%":>7}'
      f' {"Min Dist":>9} {"Max Dist":>9} {"Med Dist":>9} {"Avg Dist":>9}')
print(f'  {"-"*105}')
for _, r in stats_display.iterrows():
    print(f'  {str(r["sw_class"]):<14} {int(r["count"]):>10,} {r["pct"]:>6.1f}%'
          f'  ${r["total_usd"]:>12,.2f}  ${r["avg_usd"]:>8,.3f}'
          f'  ${r["median_usd"]:>9,.3f}  {r["win_rate"]:>5.1f}%'
          f'  {r["min_dist"]:>8.0f}  {r["max_dist"]:>8.0f}'
          f'  {r["median_dist"]:>8.1f}  {r["avg_dist"]:>8.1f}')

display(stats_display)

In [ ]:
# ── Cell 9c: Distance-bin violin plots ─────────────────────────────────

# -- Bin setup ─────────────────────────────────────────────────────────────
# BINS       = [0, 50, 100, 500, 1000, 3000, 6000, np.inf]
# BIN_LABELS = ['1-49', '50-99', '100-499', '500-999', '1000-2999', '3000-5999', '≥6000']
BINS       = [0, 500, 3000, 6000, np.inf]
BIN_LABELS = ['1-499', '500-2999', '3000-5999', '≥6000']

all_sw['dist_bin'] = pd.cut(all_sw['distance'], bins=BINS, labels=BIN_LABELS)

bin_data   = [all_sw[all_sw['dist_bin'] == lbl]['usd_profit'].dropna().values
              for lbl in BIN_LABELS]
bin_counts = [len(d) for d in bin_data]

print('Per-bin counts:')
for lbl, n in zip(BIN_LABELS, bin_counts):
    print(f'  {lbl:>12s}: {n:,}')

# -- Clip 1st–99th pct per bin for KDE stability ───────────────────────────
bin_data_clipped = []
for d in bin_data:
    if len(d) < 2:
        bin_data_clipped.append(d)
    else:
        lo, hi = np.percentile(d, [0, 100])
        bin_data_clipped.append(d[(d >= lo) & (d <= hi)])

# -- Style ─────────────────────────────────────────────────────────────────
plt.rcParams.update({'font.size': 15})  # font.family inherits global setting

BIN_COLORS = ['#4C72B0', '#55A868', '#C4A35A', '#C44E52',
               '#8172B2', '#64B5CD', '#CCB974']


In [ ]:
# ── Cell 9c-clean: Distance vs Profit ────────────────────────────────────

import matplotlib.gridspec as gridspec

_plot = all_sw[['distance', 'usd_profit']].dropna()
_plot = _plot[(_plot['distance'] > 0) & np.isfinite(_plot['usd_profit'])].copy()
_plot['usd_profit'] = _plot['usd_profit'] / 86

# ── Rolling quantile lines (computed first to determine y limits) ─────────
_sorted = _plot.sort_values('distance')
WIN  = max(1000, len(_sorted) // 10)
STEP = WIN // 8
qs      = [0.05, 0.25, 0.50, 0.75, 0.95]
q_lines = {q: [] for q in qs}
x_mids  = []
for start in range(0, len(_sorted), STEP):
    chunk = _sorted.iloc[start : start + WIN]
    if len(chunk) < 50:
        break
    x_mids.append(chunk['distance'].median())
    for q in qs:
        q_lines[q].append(chunk['usd_profit'].quantile(q))
x_mids = np.array(x_mids)

# y limits: just outside the P5–P95 band extent
_q05_min = min(q_lines[0.05])
_q95_max = max(q_lines[0.95])
_pad = (_q95_max - _q05_min) * 0.06
y_lo = _q05_min - _pad
y_hi = _q95_max + _pad*2

N_BINS = 150

def _make_fig(x_hi_val, title_suffix):
    counts_h, xedges_h = np.histogram(
        _plot['distance'], bins=N_BINS, range=(0, x_hi_val))
    xm_h = 0.5 * (xedges_h[:-1] + xedges_h[1:])

    fig_h = plt.figure(figsize=(14, 7))
    gs_h  = gridspec.GridSpec(2, 1, height_ratios=[2.5, 1], hspace=0.18)
    ax_m  = fig_h.add_subplot(gs_h[0])
    ax_c  = fig_h.add_subplot(gs_h[1], sharex=ax_m)

    ax_m.fill_between(x_mids, q_lines[0.05], q_lines[0.95],
                      alpha=0.12, color='#E55A2B', zorder=2, label='P5–P95')
    ax_m.fill_between(x_mids, q_lines[0.25], q_lines[0.75],
                      alpha=0.28, color='#E55A2B', zorder=3, label='P25–P75')
    for q, lw, ls in [(0.05,0.9,'--'),(0.95,0.9,'--'),
                      (0.25,1.2,'-'),(0.75,1.2,'-')]:
        ax_m.plot(x_mids, q_lines[q], color='#E55A2B',
                  linewidth=lw, linestyle=ls, alpha=0.6, zorder=4)
    ax_m.plot(x_mids, q_lines[0.50], color='#E55A2B',
              linewidth=2.5, zorder=5, label='Median')
    ax_m.axhline(0, color='#aaaaaa', linewidth=1.0, zorder=6)
    ax_m.set_xlim(0, x_hi_val)
    ax_m.set_ylim(y_lo, y_hi)
    ax_m.set_ylabel('Sandwich\nProfit (USD/86)', fontsize=30)
    ax_m.tick_params(axis='x', labelbottom=False, length=0, labelsize=24)
    ax_m.tick_params(axis='y', labelsize=24)
    ax_m.yaxis.set_major_formatter(make_sci_fmt())
    ax_m.yaxis.get_offset_text().set_fontsize(20)
    ax_m.set_axisbelow(True)
    ax_m.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.35, color='#bbbbbb')
    ax_m.legend(title='Profit percentiles', title_fontsize=24,
               prop={'size': 22}, framealpha=0.9, loc='upper right', ncol=3)

    ax_c.fill_between(xm_h, counts_h, alpha=0.4, color='#3B5BA5', zorder=2)
    ax_c.plot(xm_h, counts_h, color='#3B5BA5', linewidth=1.5, zorder=3)
    ax_c.set_xlabel('Distance', fontsize=30)
    ax_c.set_ylabel('Sandwich\nCount', fontsize=30)
    ax_c.tick_params(axis='x', labelsize=24)
    ax_c.tick_params(axis='y', labelsize=24)
    ax_c.yaxis.set_major_formatter(make_sci_fmt())
    ax_c.yaxis.get_offset_text().set_fontsize(22)
    ax_c.set_axisbelow(True)
    ax_c.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.3, color='#bbbbbb')
    return fig_h

x_hi_p99 = np.nanpercentile(_plot['distance'], 99.95)
x_hi_max = _plot['distance'].max()

fig_p99 = _make_fig(x_hi_p99, 'x: P99 cutoff')
fig_max = _make_fig(x_hi_max, 'x: full range')

# ── Split figure: in-block vs cross-block ────────────────────────────────
_IB_COL = '#3B5BA5'
_CB_COL = '#E55A2B'

def _rolling_qs(df_sub, win, step, qs):
    _s = df_sub.sort_values('distance')
    q_lines_ = {q: [] for q in qs}
    x_mids_  = []
    for start in range(0, len(_s), step):
        chunk = _s.iloc[start : start + win]
        if len(chunk) < 50:
            break
        x_mids_.append(chunk['distance'].median())
        for q in qs:
            q_lines_[q].append(chunk['usd_profit'].quantile(q))
    return np.array(x_mids_), q_lines_

_plot_ib = all_sw[all_sw['sw_class'] == 'inblock'][['distance', 'usd_profit']].dropna()
_plot_ib = _plot_ib[(_plot_ib['distance'] > 0) & np.isfinite(_plot_ib['usd_profit'])].copy()
_plot_ib['usd_profit'] = _plot_ib['usd_profit'] / 86

_plot_cb = all_sw[all_sw['sw_class'] == 'cross_block'][['distance', 'usd_profit']].dropna()
_plot_cb = _plot_cb[(_plot_cb['distance'] > 0) & np.isfinite(_plot_cb['usd_profit'])].copy()
_plot_cb['usd_profit'] = _plot_cb['usd_profit'] / 86

WIN_IB  = max(500, len(_plot_ib) // 10)
STEP_IB = max(1, WIN_IB // 8)
WIN_CB  = max(500, len(_plot_cb) // 10)
STEP_CB = max(1, WIN_CB // 8)

xm_ib, ql_ib = _rolling_qs(_plot_ib, WIN_IB, STEP_IB, qs)
xm_cb, ql_cb = _rolling_qs(_plot_cb, WIN_CB, STEP_CB, qs)

_y_lo2 = min(min(ql_ib[0.05]), min(ql_cb[0.05]))
_y_hi2 = max(max(ql_ib[0.95]), max(ql_cb[0.95]))
_pad2  = (_y_hi2 - _y_lo2) * 0.04
y_lo2  = _y_lo2 - _pad2
y_hi2  = _y_hi2 + _pad2

counts_ib, xedges_ib = np.histogram(_plot_ib['distance'], bins=N_BINS, range=(0, x_hi_p99))
counts_cb, xedges_cb = np.histogram(_plot_cb['distance'], bins=N_BINS, range=(0, x_hi_p99))
xm_hist_ib = 0.5 * (xedges_ib[:-1] + xedges_ib[1:])
xm_hist_cb = 0.5 * (xedges_cb[:-1] + xedges_cb[1:])

def _draw_profit(ax, xm_, ql_, col_, title):
    ax.fill_between(xm_, ql_[0.05], ql_[0.95], alpha=0.10, color=col_, zorder=2)
    ax.fill_between(xm_, ql_[0.25], ql_[0.75], alpha=0.22, color=col_, zorder=3)
    ax.plot(xm_, ql_[0.50], color=col_, linewidth=2.5, zorder=5)
    ax.axhline(0, color='#aaaaaa', linewidth=1.0, zorder=6)
    ax.set_xlim(0, x_hi_p99)
    ax.set_ylim(y_lo2, y_hi2)
    ax.set_ylabel('Sandwich\nProfit (SOL)', fontsize=28)
    ax.tick_params(axis='y', labelsize=22)
    ax.yaxis.set_major_formatter(make_sci_fmt())
    ax.yaxis.get_offset_text().set_fontsize(18)
    ax.set_axisbelow(True)
    ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.35, color='#bbbbbb')
    ax.set_title(title, fontsize=26, pad=8)
    _h = [
        Patch(facecolor=col_, alpha=0.10, edgecolor=col_, label='P5–P95'),
        Patch(facecolor=col_, alpha=0.22, edgecolor=col_, label='P25–P75'),
        _L2D([0],[0], color=col_, linewidth=2.5,          label='Median'),
    ]
    ax.legend(handles=_h, prop={'size': 18}, framealpha=0.9, loc='upper right')

# ── Layout A: two stacked profit panels, no count ────────────────────────
fig_stacked, (ax_ib, ax_cb) = plt.subplots(2, 1, figsize=(14, 9),
                                            sharex=True, sharey=True)
fig_stacked.subplots_adjust(hspace=0.12)
_draw_profit(ax_ib, xm_ib, ql_ib, _IB_COL,  'In-block')
_draw_profit(ax_cb, xm_cb, ql_cb, _CB_COL, 'Cross-block')
ax_ib.tick_params(axis='x', labelbottom=False, length=0)
ax_cb.set_xlabel('Distance', fontsize=28)
ax_cb.tick_params(axis='x', labelsize=22)

# ── Layout B: two profit panels side-by-side + wide count panel ──────────
fig_2col = plt.figure(figsize=(18, 8))
gs_2col  = gridspec.GridSpec(2, 2, height_ratios=[2.5, 1], hspace=0.15, wspace=0.06)
ax_ib2   = fig_2col.add_subplot(gs_2col[0, 0])
ax_cb2   = fig_2col.add_subplot(gs_2col[0, 1], sharey=ax_ib2)
ax_cnt2  = fig_2col.add_subplot(gs_2col[1, :])

_draw_profit(ax_ib2, xm_ib, ql_ib, _IB_COL,  'In-block')
_draw_profit(ax_cb2, xm_cb, ql_cb, _CB_COL, 'Cross-block')
ax_ib2.tick_params(axis='x', labelbottom=False, length=0)
ax_cb2.tick_params(axis='x', labelbottom=False, length=0)
ax_cb2.tick_params(axis='y', labelleft=False)

for xm_h, cnts, col_, lbl_ in [
    (xm_hist_ib, counts_ib, _IB_COL, 'In-block'),
    (xm_hist_cb, counts_cb, _CB_COL, 'Cross-block'),
]:
    ax_cnt2.fill_between(xm_h, cnts, alpha=0.30, color=col_, zorder=2, label=lbl_)
    ax_cnt2.plot(xm_h, cnts, color=col_, linewidth=1.5, zorder=3)
ax_cnt2.set_xlabel('Distance', fontsize=28)
ax_cnt2.set_ylabel('Sandwich\nCount', fontsize=28)
ax_cnt2.tick_params(labelsize=20)
ax_cnt2.yaxis.set_major_formatter(make_sci_fmt())
ax_cnt2.yaxis.get_offset_text().set_fontsize(18)
ax_cnt2.set_axisbelow(True)
ax_cnt2.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.3, color='#bbbbbb')
ax_cnt2.legend(prop={'size': 18}, framealpha=0.9, loc='upper right')
ax_cnt2.set_xlim(0, x_hi_p99)

pdf_path = 'sandwich_distance_clean.pdf'
with PdfPages(pdf_path) as pdf:
    pdf.savefig(fig_p99,     bbox_inches='tight')
    pdf.savefig(fig_max,     bbox_inches='tight')
    pdf.savefig(fig_stacked, bbox_inches='tight')
    pdf.savefig(fig_2col,    bbox_inches='tight')
plt.figure(fig_p99.number);     plt.show()
plt.figure(fig_max.number);     plt.show()
plt.figure(fig_stacked.number); plt.show()
plt.figure(fig_2col.number);    plt.show()
print(f'Saved: {pdf_path}')


In [ ]:
# ── Cell 9d: Distance-bin summary table ─────────────────────────────────

total = sum(bin_counts)
rows = []
for lbl, data, n in zip(BIN_LABELS, bin_data, bin_counts):
    rows.append({
        'Distance Bin':   lbl,
        'Count':          n,
        'Pct (%)':        round(n / total * 100, 2) if total else 0,
        'Avg Profit ($)': round(float(data.mean()), 4) if len(data) else float('nan'),
        'Min Profit ($)':  round(float(data.min()),  4) if len(data) else float('nan'),
        'Max Profit ($)':  round(float(data.max()),  4) if len(data) else float('nan'),
        'Median Profit ($)': round(float(pd.Series(data).median()), 4) if len(data) else float('nan'),
        'Pct Profitable (%)': round((data >= 0).sum() / len(data) * 100, 2) if len(data) else float('nan'),
        'In-block (%)': round(
            (all_sw[all_sw['dist_bin'] == lbl]['sw_class'] != 'cross_block').mean() * 100, 2
        ),
    })

bin_summary = pd.DataFrame(rows)
display(bin_summary)


In [ ]:
# ── Cell: Adverse-tx count per sandwich — in-block vs cross-block ─────────
print('Querying adverse tx counts from sandwich_txs...')
adverse_df = client.query_df("""
SELECT
    st.sandwichId,
    countIf(st.type = 'adverse') AS adverse_count
FROM sandwich_txs st
INNER JOIN sandwiches s ON st.sandwichId = s.sandwichId
WHERE s.slot >= {START_SLOT} AND s.slot < {END_SLOT}
GROUP BY st.sandwichId
HAVING adverse_count > 0
""".format(START_SLOT=START_SLOT, END_SLOT=END_SLOT))
print(f'  Sandwiches with >=1 adverse tx: {len(adverse_df):,}')

# Merge with all_sw to get sw_class
adv = all_sw[['sandwichId', 'sw_class']].merge(adverse_df, on='sandwichId', how='left')
adv['adverse_count'] = adv['adverse_count'].fillna(0).astype(int)

SEP = '=' * 60
print(SEP)
print('  ADVERSE TX COUNT PER SANDWICH')
print(SEP)

rows = []
for cls, label in [('inblock', 'In-block'), ('cross_block', 'Cross-block')]:
    sub     = adv[adv['sw_class'] == cls]['adverse_count']
    nonzero = sub[sub > 0]
    rows.append({
        'Class':               label,
        'Total sandwiches':    len(sub),
        'With adverse tx':     len(nonzero),
        'With adverse (%)':    f'{100*len(nonzero)/max(len(sub),1):.1f}%',
        'Min (all)':           int(sub.min()),
        'Max (all)':           int(sub.max()),
        'Median (all)':        round(sub.median(), 1),
        'Mean (all)':          round(sub.mean(),   2),
        'Median (>0)':         round(nonzero.median(), 1) if len(nonzero) else 0,
        'Mean (>0)':           round(nonzero.mean(),   2) if len(nonzero) else 0,
    })
    print(f'\n  {label}:')
    print(f'    Total sandwiches:  {len(sub):>8,}')
    print(f'    With adverse tx:   {len(nonzero):>8,}  ({100*len(nonzero)/max(len(sub),1):.1f}%)')
    print(f'    Min (all):         {int(sub.min()):>8}')
    print(f'    Max (all):         {int(sub.max()):>8}')
    print(f'    Median (all):      {sub.median():>8.1f}')
    print(f'    Mean   (all):      {sub.mean():>8.2f}')
    if len(nonzero):
        print(f'    Median (>0 only):  {nonzero.median():>8.1f}')
        print(f'    Mean   (>0 only):  {nonzero.mean():>8.2f}')

print(f'\n{SEP}')
display(pd.DataFrame(rows))

# ── Win rate by class × adverse presence ─────────────────────────────────
adv_full = all_sw[['sandwichId', 'sw_class', 'usd_profit']].merge(
    adverse_df, on='sandwichId', how='left'
)
adv_full['adverse_count'] = adv_full['adverse_count'].fillna(0).astype(int)
adv_full['has_adverse']   = adv_full['adverse_count'] > 0
adv_full['win']           = adv_full['usd_profit'] > 0

wr_rows = []
for cls, cls_label in [('inblock', 'In-block'), ('cross_block', 'Cross-block')]:
    for has_adv, adv_label in [(True, 'w/ adverse'), (False, 'w/o adverse')]:
        mask = (adv_full['sw_class'] == cls) & (adv_full['has_adverse'] == has_adv)
        grp  = adv_full[mask]
        wr   = grp['win'].mean() * 100 if len(grp) else float('nan')
        wr_rows.append({
            'Class':        cls_label,
            'Adverse':      adv_label,
            'Count':        len(grp),
            'Win rate (%)': round(wr, 2),
        })

print('\n  WIN RATE BY CLASS x ADVERSE PRESENCE')
print(SEP)
display(pd.DataFrame(wr_rows))

# ── Adverse count (median) by distance bin ────────────────────────────────
adv_dist = all_sw[['sandwichId', 'dist_bin']].merge(
    adverse_df, on='sandwichId', how='left'
)
adv_dist['adverse_count'] = adv_dist['adverse_count'].fillna(0).astype(int)

adv_dist2 = adv_dist.merge(all_sw[['sandwichId', 'sw_class']], on='sandwichId', how='left')
total_cross = (adv_dist2['sw_class'] == 'cross_block').sum()

dist_rows = []
for lbl in BIN_LABELS:
    mask    = adv_dist2['dist_bin'] == lbl
    grp_adv = adv_dist2[mask]['adverse_count']
    nonzero = grp_adv[grp_adv > 0]
    n_cross = (adv_dist2[mask]['sw_class'] == 'cross_block').sum()
    n_total = mask.sum()
    dist_rows.append({
        'Distance bin':         lbl,
        'Count':                n_total,
        'Cross-block in bin (%)': f'{100*n_cross/max(n_total,1):.1f}%',
        '% of all cross-block':  f'{100*n_cross/max(total_cross,1):.1f}%',
        'With adverse (%)':     f'{100*len(nonzero)/max(n_total,1):.1f}%',
        'Median adverse (all)': round(grp_adv.median(), 1),
        'Median adverse (>0)':  round(nonzero.median(), 1) if len(nonzero) else '-',
    })

print('\n  ADVERSE COUNT BY DISTANCE BIN')
print(SEP)
display(pd.DataFrame(dist_rows))

# ── Victim count by distance bin ─────────────────────────────────────────
vic_dist_rows = []
for lbl in BIN_LABELS:
    grp = all_sw[all_sw['dist_bin'] == lbl]['victim_count']
    vic_dist_rows.append({
        'Distance bin':   lbl,
        'Count':          len(grp),
        'Min':            int(grp.min()),
        'Max':            int(grp.max()),
        'Median':         round(grp.median(), 1),
        'Mean':           round(grp.mean(), 2),
        'Total victims':  int(grp.sum()),
    })

print('\n  VICTIM COUNT BY DISTANCE BIN')
print(SEP)
display(pd.DataFrame(vic_dist_rows))


---

## Section 4: Jito Bundle Usage

Classifies sandwiches by Jito bundle membership pattern across tx types (frontRun / victim / backRun).

| Pattern | Definition |
|---------|-----------|
| **full** | All front-run(s), victim(s) and back-run(s) share the **same** bundle |
| **fr + victim** | Front-run(s) and victim(s) bundled together; back-run not |
| **br + victim** | Back-run(s) and victim(s) bundled together; front-run not |
| **fr + br** | Front and back bundled (possibly same or different bundles); victims not |
| **fr only** | Only front-run(s) in a bundle |
| **br only** | Only back-run(s) in a bundle |
| **no bundle** | No Jito bundle used |

In [ ]:
# ── Cell 9a: Classification + Distance ───────────────────────────────────

# 1. Classify from existing parquet columns (no DB needed)
conditions = [
    all_sw['consecutive'],
    ~all_sw['consecutive'] & all_sw['cross_block'],
]
all_sw['sw_class'] = np.select(conditions, ['consecutive', 'cross_block'], default='inblock')
all_sw['sw_class'] = all_sw['sw_class'].astype('category')
print('Class distribution:')
print(all_sw['sw_class'].value_counts().to_string())

# 2. Query DB: first-front and last-back (slot, position) per sandwich
# Distance = # transactions between first front run and last back run
print('\nQuerying first-front / last-back positions from DB...')
dist_df = client.query_df(f"""
SELECT
    st.sandwichId,
    argMinIf(toUInt64(st.slot),    toUInt64(st.slot)*100000 + toUInt64(st.position), st.type = 'frontRun') AS ff_slot,
    argMinIf(toInt32(st.position), toUInt64(st.slot)*100000 + toUInt64(st.position), st.type = 'frontRun') AS ff_pos,
    argMaxIf(toUInt64(st.slot),    toUInt64(st.slot)*100000 + toUInt64(st.position), st.type = 'backRun')  AS lb_slot,
    argMaxIf(toInt32(st.position), toUInt64(st.slot)*100000 + toUInt64(st.position), st.type = 'backRun')  AS lb_pos
FROM sandwich_txs st
INNER JOIN sandwiches s ON st.sandwichId = s.sandwichId
WHERE s.slot >= {START_SLOT} AND s.slot < {END_SLOT}
  AND st.type IN ('frontRun', 'backRun')
GROUP BY st.sandwichId
""")
print(f'  {len(dist_df):,} sandwiches with position data')
for col in ['ff_slot', 'lb_slot']:
    dist_df[col] = dist_df[col].astype(int)
for col in ['ff_pos', 'lb_pos']:
    dist_df[col] = dist_df[col].astype(int)

# 3. Same-block distance (vectorized): last_back_pos - first_front_pos - 1
same_mask = dist_df['ff_slot'] == dist_df['lb_slot']
dist_df['distance'] = np.where(
    same_mask,
    dist_df['lb_pos'] - dist_df['ff_pos'] - 1,
    np.nan
)

# 4. Cross-block distance via slot_txs
cross = dist_df[~same_mask]
print(f'  Cross-block sandwiches: {len(cross):,}')

if len(cross) > 0:
    min_s, max_s = int(cross['ff_slot'].min()), int(cross['lb_slot'].max())
    stx = client.query_df(
        f'SELECT slot, txCount FROM slot_txs WHERE slot >= {min_s} AND slot <= {max_s}'
    )
    stx_map = dict(zip(stx['slot'].astype(int), stx['txCount'].astype(int)))
    slot_series = pd.Series(stx_map, dtype=int).sort_index()

    def _cross_dist(row):
        ff_s, ff_p = int(row['ff_slot']), int(row['ff_pos'])
        lb_s, lb_p = int(row['lb_slot']), int(row['lb_pos'])
        d = stx_map.get(ff_s, 0) - ff_p - 1          # txs after first_front in ff_slot
        if lb_s > ff_s + 1:
            d += int(slot_series.loc[ff_s + 1: lb_s - 1].sum())  # full intermediate slots
        d += lb_p                                       # txs before last_back in lb_slot
        return max(d, 0)

    cross_dists = cross.apply(_cross_dist, axis=1)
    dist_df.loc[~same_mask, 'distance'] = cross_dists.values

dist_df['distance'] = dist_df['distance'].fillna(0).clip(lower=0)

# 5. Merge back to all_sw
if 'distance' in all_sw.columns:
    all_sw = all_sw.drop(columns=['distance'])
all_sw = all_sw.merge(dist_df[['sandwichId', 'distance']], on='sandwichId', how='left')
print(f'\nDistance coverage: {all_sw["distance"].notna().sum():,} / {len(all_sw):,}')
print(f'Distance — min:{all_sw["distance"].min():.0f}  '
      f'mean:{all_sw["distance"].mean():.1f}  '
      f'median:{all_sw["distance"].median():.0f}  '
      f'max:{all_sw["distance"].max():.0f}')


In [ ]:
# ── Cell 11: Jito bundle classification (data-first refactor) ────────────

def _norm_sid(df):
    if 'st.sandwichId' in df.columns:
        if 'sandwichId' in df.columns:
            df = df.drop(columns=['sandwichId'])
        return df.rename(columns={'st.sandwichId': 'sandwichId'})
    if 'sandwichId' not in df.columns:
        df.index.name = 'sandwichId'
        return df.reset_index()
    return df

# ── Step 1: find sandwiches in all_sw that have any bundled fr or br ──────
print('Step 1: finding sandwiches with bundled fr/br...')
bun_sids_df = client.query_df(f"""
SELECT DISTINCT st.sandwichId
FROM sandwich_txs st
INNER JOIN sandwiches s ON st.sandwichId = s.sandwichId
WHERE s.slot >= {START_SLOT} AND s.slot < {END_SLOT}
  AND st.type IN ('frontRun', 'backRun')
  AND st.inBundle = 1
""")
all_sw_sids = set(all_sw['sandwichId'])
bundle_sids = [sid for sid in bun_sids_df['sandwichId'] if sid in all_sw_sids]
print(f'  {len(bundle_sids):,} sandwiches in all_sw have bundled fr/br')

# ── Step 2: fetch all txs for those sandwiches only ───────────────────────
print('\nStep 2: fetching all txs for bundle sandwiches...')
sid_str = "','".join(bundle_sids)
txs_df = client.query_df(f"""
SELECT st.sandwichId, st.type, st.signature, st.slot, st.inBundle
FROM sandwich_txs st
WHERE st.sandwichId IN ('{sid_str}')
  AND st.type IN ('frontRun', 'backRun', 'victim')
""", settings={'max_query_size': 10_000_000})
print(f'  {len(txs_df):,} txs')

# ── Step 3: resolve bundleId via jito_bundles ─────────────────────────────
print('\nStep 3: resolving bundleIds...')
slots_str = ','.join(map(str, txs_df.loc[txs_df['inBundle'], 'slot'].unique()))
jb_df = client.query_df(f"""
SELECT bundleId, arrayJoin(transactions) AS sig
FROM jito_bundles WHERE slot IN ({slots_str})
""")
sig_to_bundle = dict(zip(jb_df['sig'], jb_df['bundleId']))
print(f'  {len(sig_to_bundle):,} sig→bundleId entries')

txs_df['bundleId'] = txs_df['signature'].map(sig_to_bundle)

# ── Step 4: per-sandwich bundleId sets → flags ────────────────────────────
print('\nStep 4: computing bundle flags...')

def _bundle_sets(grp):
    fr  = set(grp.loc[grp['type'] == 'frontRun', 'bundleId'].dropna())
    br  = set(grp.loc[grp['type'] == 'backRun',  'bundleId'].dropna())
    vic = set(grp.loc[grp['type'] == 'victim',   'bundleId'].dropna())
    return pd.Series({
        'fr_b':        len(fr)  > 0,
        'br_b':        len(br)  > 0,
        'vic_b':       len(vic) > 0,
        'fr_br_same':  bool(fr & br),
        'fr_vic_same': bool(fr & vic),
        'br_vic_same': bool(br & vic),
        'all_same':    bool(fr & vic & br),
    })

bundle_flags = txs_df.groupby('sandwichId').apply(_bundle_sets).reset_index()
for col in bundle_flags.columns[1:]:
    bundle_flags[col] = bundle_flags[col].astype(bool)

# ── Step 5: coarse / fine labels ─────────────────────────────────────────
def _coarse(row):
    if not row.fr_b and not row.br_b: return 'No bundle'
    if row.fr_br_same:                return 'Same bundle (fr+br)'
    return 'Partial / mixed bundle'

def _fine(row):
    if not row.fr_b and not row.br_b: return 'No bundle'
    if row.all_same:                  return 'fr+br+victim (same bundle)'
    if row.fr_br_same:                return 'fr+br (same bundle, no victim)'
    if row.fr_vic_same:               return 'fr+victim (same bundle)'
    if row.br_vic_same:               return 'br+victim (same bundle)'
    return 'fr/br only (no victim in bundle)'

bundle_flags['coarse'] = bundle_flags.apply(_coarse, axis=1)
bundle_flags['fine']   = bundle_flags.apply(_fine,   axis=1)

# ── Step 6: merge into all_sw (non-bundle sandwiches → No bundle) ─────────
DROP_COLS = ['fr_b','br_b','vic_b','fr_br_same','fr_vic_same','br_vic_same',
             'all_same','coarse','fine','jito_pattern','bundle_tier']
all_sw = all_sw.drop(columns=[c for c in DROP_COLS if c in all_sw.columns])
all_sw = all_sw.merge(bundle_flags, on='sandwichId', how='left')
for col in ['fr_b','br_b','vic_b','fr_br_same','fr_vic_same','br_vic_same','all_same']:
    all_sw[col] = all_sw[col].fillna(False).astype(bool)
all_sw['coarse'] = all_sw['coarse'].fillna('No bundle')
all_sw['fine']   = all_sw['fine'].fillna('No bundle')
print(f'  Merged. Total: {len(all_sw):,}')

# ── Print stats ───────────────────────────────────────────────────────────
def _stats_table(df, col, title):
    grp = (
        df.groupby(col, observed=True)
        .agg(count=('sandwichId','count'),
             avg_usd=('usd_profit','mean'),
             median_usd=('usd_profit', lambda x: x.median()))
        .reset_index()
    )
    grp['pct'] = grp['count'] / len(df) * 100
    grp = grp.sort_values('count', ascending=False).reset_index(drop=True)
    total = pd.DataFrame([{col: 'TOTAL', 'count': len(df), 'pct': 100.0,
                           'avg_usd': df['usd_profit'].mean(),
                           'median_usd': df['usd_profit'].median()}])
    out = pd.concat([grp, total], ignore_index=True)
    print(f'\n{"="*70}')
    print(f'  {title}')
    print(f'{"="*70}')
    print(f'  {col:<42} {"Count":>8} {"Pct%":>6} {"Avg $":>8} {"Med $":>8}')
    print(f'  {"-"*74}')
    for _, r in out.iterrows():
        print(f'  {str(r[col]):<42} {int(r["count"]):>8,} {r["pct"]:>5.1f}%'
              f' {r["avg_usd"]:>8.3f} {r["median_usd"]:>8.3f}')
    display(out)

_stats_table(all_sw, 'coarse', 'BUNDLE — COARSE')
_stats_table(all_sw, 'fine',   'BUNDLE — FINE')


In [ ]:
# ── Cell 11b: fr+br+victim (same bundle) — per-signer breakdown ──────────

subset = all_sw[all_sw['fine'] == 'fr+br+victim (same bundle)'].copy()
total  = len(subset)
print(f'fr+br+victim (same bundle): {total:,} sandwiches, '
      f'{subset["signer"].nunique():,} signers')

stats = (
    subset.groupby('signer', observed=True)
    .agg(
        count     = ('sandwichId', 'count'),
        total_usd = ('usd_profit', 'sum'),
        avg_usd   = ('usd_profit', 'mean'),
        win_rate  = ('usd_profit', lambda x: (x > 0).mean() * 100),
    )
    .reset_index()
)
stats['pct'] = stats['count'] / total * 100
stats = stats.sort_values('count', ascending=False).reset_index(drop=True)

print(f'\n  {"Signer":<46} {"Count":>8} {"Pct%":>6} {"Total USD":>12} {"Avg USD":>9} {"WinRate%":>9}')
print(f'  {"-"*95}')
for _, r in stats.iterrows():
    print(f'  {str(r["signer"]):<46} {int(r["count"]):>8,} {r["pct"]:>5.2f}%'
          f' ${r["total_usd"]:>11,.2f} ${r["avg_usd"]:>8,.3f} {r["win_rate"]:>8.1f}%')

display(stats[['signer', 'count', 'pct', 'total_usd', 'avg_usd', 'win_rate']])
show(all_sw[all_sw['jito_bundle']==True]['usd_profit'].describe())


In [ ]:
# ── Cell 11b: List same-bundle sandwiches ────────────────────────────────

same_bundle_sw = all_sw[all_sw['fr_br_same']].copy()
print(f'Same-bundle sandwiches: {len(same_bundle_sw):,}')

cols = ['sandwichId', 'slot', 'signer', 'sw_class',
        'victim_count', 'front_count', 'back_count',
        'usd_profit', 'fine']
display_cols = [c for c in cols if c in same_bundle_sw.columns]
display(same_bundle_sw[display_cols].sort_values('usd_profit', ascending=False).reset_index(drop=True))


In [ ]:
# ── Cell 11c: Victim txs with Jito dontfront tip account ─────────────────
# Counts victim transactions that included the dontfront account key
# (indicating the sender tried to opt out of front-running) but were
# sandwiched anyway.

DONTFRONT = 'jitodontfront111111111111111111111111111111'

sw_sids_str = "','".join(all_sw['sandwichId'].tolist())

print('Querying victim txs with dontfront account key...')
df_df = client.query_df(
    f"""
    SELECT
        st.sandwichId,
        st.signature,
        st.slot
    FROM sandwich_txs st
    WHERE st.sandwichId IN ('{sw_sids_str}')
      AND st.type = 'victim'
      AND has(st.accountKeys, 'jitodontfront111111111111111111111111111111')
    """,
    settings={'max_query_size': 10_000_000}
)
print(f'  Done.')

n_txs       = len(df_df)
n_sandwiches = df_df['sandwichId'].nunique()

SEP = '=' * 55
print(SEP)
print('  VICTIMS WITH JITO DONTFRONT ACCOUNT KEY')
print(SEP)
print(f'  Victim txs with dontfront:  {n_txs:>8,}')
print(f'  Sandwiches affected:         {n_sandwiches:>8,}')
print(f'  Total victim txs in all_sw:  {len(all_sw):>8,}  (sandwich count)')
print(f'  Pct of sandwiches:           {100*n_sandwiches/len(all_sw):>7.2f}%')
print(SEP)

if n_txs > 0:
    display(df_df.head(10))

# Total profit of affected sandwiches
if n_txs > 0:
    total_profit = all_sw.loc[
        all_sw['sandwichId'].isin(df_df['sandwichId']), 'usd_profit'
    ].sum()
    print(f'  Total profit of affected sandwiches: ${total_profit:,.2f} USD')
    print(SEP)


In [ ]:
# ── Cell 11c-overlap: dontfront sandwiches ∩ same-bundle sandwiches ──────

dontfront_ids   = set(df_df['sandwichId'])
same_bundle_ids = set(same_bundle_sw['sandwichId'])

overlap_ids = dontfront_ids & same_bundle_ids

SEP = '=' * 55
print(SEP)
print('  OVERLAP: DONTFRONT  vs  SAME-BUNDLE')
print(SEP)
print(f'  Dontfront sandwiches:        {len(dontfront_ids):>8,}')
print(f'  Same-bundle sandwiches:      {len(same_bundle_ids):>8,}')
print(f'  Overlap (in both):           {len(overlap_ids):>8,}')
print(f'  Overlap / dontfront:         {100*len(overlap_ids)/max(len(dontfront_ids),1):>7.2f}%')
print(f'  Overlap / same-bundle:       {100*len(overlap_ids)/max(len(same_bundle_ids),1):>7.2f}%')
print(SEP)

if overlap_ids:
    overlap_df = all_sw[all_sw['sandwichId'].isin(overlap_ids)].copy()
    print(f'\n  Overlap sandwiches (sorted by profit):')
    cols = ['sandwichId', 'slot', 'signer', 'sw_class',
            'victim_count', 'front_count', 'back_count', 'usd_profit', 'fine']
    display_cols = [c for c in cols if c in overlap_df.columns]
    display(overlap_df[display_cols].sort_values('usd_profit', ascending=False).reset_index(drop=True))
    print(f'\n  Total profit of overlap: ${overlap_df["usd_profit"].sum():,.2f} USD')
else:
    print('\n  No overlap — dontfront and same-bundle sets are disjoint.')


---

## Section 5: Fee & Cost Analysis

Per-sandwich breakdown of transaction fees (frontRun / victim / backRun) and Jito bundle tips. All aggregation is done server-side in ClickHouse to avoid loading per-tx rows into memory.

In [ ]:
# ── Cell Fee-1: Per-sandwich fee sums by type (server-side) ──────────────
# Scoped strictly to sandwiches in all_sw (post-filter set)
sw_sids     = all_sw['sandwichId'].tolist()
sw_sids_str = "','".join(sw_sids)
print(f'all_sw sandwiches: {len(sw_sids):,}')

print('Querying per-type fee sums (server-side aggregation)...')
fee_sums = client.query_df(
    f"""
    SELECT
        st.sandwichId,
        sumIf(st.fee, st.type = 'frontRun') AS fr_fee,
        sumIf(st.fee, st.type = 'victim')   AS victim_fee,
        sumIf(st.fee, st.type = 'backRun')  AS br_fee
    FROM sandwich_txs st
    WHERE st.sandwichId IN ('{sw_sids_str}')
      AND st.type IN ('frontRun', 'victim', 'backRun')
    GROUP BY st.sandwichId
    """,
    settings={'max_query_size': 10_000_000}
)
print(f'  {len(fee_sums):,} sandwiches returned')
print(fee_sums[['fr_fee', 'victim_fee', 'br_fee']].describe())


In [ ]:
# ── Cell Fee-2: Bundle tips for sandwiches with bundled fr/br ──────────
bsids = all_sw.loc[all_sw['fr_b'] | all_sw['br_b'], 'sandwichId'].tolist()
print(f'Sandwiches with bundled fr/br: {len(bsids):,}')

tip_df2 = pd.DataFrame(columns=['sandwichId', 'tip_lamports'])

if bsids:
    sid_str = "','".join(bsids)

    # Step 1: get inBundle fr/br sigs for these sandwiches
    sig_df2 = client.query_df(
        f"""SELECT st.sandwichId, st.signature, st.slot
FROM sandwich_txs st
WHERE st.sandwichId IN ('{sid_str}')
  AND st.type IN ('frontRun', 'backRun')
  AND st.inBundle = 1""",
        settings={'max_query_size': 10_000_000})
    print(f'  {len(sig_df2):,} inBundle fr/br txs')

    # Step 2: resolve bundleId + tip from jito_bundles
    slots_str = ','.join(map(str, sig_df2['slot'].unique()))
    jb2 = client.query_df(f"""
SELECT bundleId, arrayJoin(transactions) AS sig, landedTipLamports AS tip_lamports
FROM jito_bundles WHERE slot IN ({slots_str})
""")
    sig_df2 = sig_df2.merge(jb2, left_on='signature', right_on='sig', how='left')

    # Step 3: dedup by (sandwichId, bundleId), then sum tips per sandwich
    tip_df2 = (
        sig_df2.dropna(subset=['bundleId'])
        .drop_duplicates(subset=['sandwichId', 'bundleId'])
        .groupby('sandwichId', as_index=False)['tip_lamports'].sum()
    )
    print(f'  {len(tip_df2):,} sandwiches with tip data')
    print(f'  tip_lamports — '
          f'mean: {tip_df2["tip_lamports"].mean():,.0f}  '
          f'total: {tip_df2["tip_lamports"].sum():,.0f}')


In [ ]:
# ── Cell Fee-3: Total attacker cost summary ─────────────────────────────

# Reuse sol_price from Cell 16 if available, else re-read from CSV
if 'sol_price' not in dir():
    _prices = pd.read_csv(PRICES_CSV)
    _sol_row = _prices[_prices['token'] == SOL_ADDR]
    sol_price = float(_sol_row['usd_price'].iloc[0]) if len(_sol_row) else 0.0
print(f'SOL price: ${sol_price:.4f}')

sw_cost = fee_sums.copy()
sw_cost = sw_cost.merge(tip_df2, on='sandwichId', how='left')
sw_cost['tip_lamports'] = sw_cost['tip_lamports'].fillna(0)
sw_cost['attacker_cost_lamports'] = (sw_cost['fr_fee']
                                     + sw_cost['br_fee']
                                     + sw_cost['tip_lamports'])

for col_l, col_u in [
    ('fr_fee',                   'fr_fee_usd'),
    ('victim_fee',               'victim_fee_usd'),
    ('br_fee',                   'br_fee_usd'),
    ('tip_lamports',             'tip_usd'),
    ('attacker_cost_lamports',   'attacker_cost_usd'),
]:
    sw_cost[col_u] = sw_cost[col_l] / 1e9

totals = sw_cost[['fr_fee_usd', 'victim_fee_usd', 'br_fee_usd',
                   'tip_usd', 'attacker_cost_usd']].sum()
means  = sw_cost[['fr_fee_usd', 'victim_fee_usd', 'br_fee_usd',
                   'tip_usd', 'attacker_cost_usd']].mean()
# tip mean only over sandwiches that actually paid a tip
mean_tip_with_tip = sw_cost.loc[sw_cost['tip_usd'] > 0, 'tip_usd'].mean()
n_with_tip = (sw_cost['tip_usd'] > 0).sum()

SEP = '=' * 78
print(f'\n{SEP}')
print('  TOTAL ATTACKER COST  (fr_fee + br_fee + tip)')
print(SEP)
print(f'  {"Component":<20} {"Total USD":>14} {"Mean USD / sw":>16}')
print(f'  {"-"*52}')
for label, tot, mn in [
    ('fr fee',         totals['fr_fee_usd'],       means['fr_fee_usd']),
    ('victim fee',     totals['victim_fee_usd'],    means['victim_fee_usd']),
    ('br fee',         totals['br_fee_usd'],        means['br_fee_usd']),
    ('attacker total', totals['attacker_cost_usd'], means['attacker_cost_usd']),
]:
    print(f'  {label:<20} {tot:>13,.2f} {mn:>15,.4f}')
print(f'  {"tip (bundle)":<20} {totals["tip_usd"]:>13,.2f}'
      f' {mean_tip_with_tip:>14,.4f}  (n={n_with_tip:,} w/ tip)')
print(SEP)
display(sw_cost[['sandwichId', 'fr_fee_usd', 'victim_fee_usd', 'br_fee_usd',
                  'tip_usd', 'attacker_cost_usd']].describe())

# ── Per-tx fee averages ──────────────────────────────────────────────────
_pt = sw_cost[['sandwichId', 'fr_fee', 'br_fee']].merge(
    all_sw[['sandwichId', 'front_count', 'back_count']], on='sandwichId'
)
_pt['fr_fee_sol'] = _pt['fr_fee'] / 1e9
_pt['br_fee_sol'] = _pt['br_fee'] / 1e9
total_fr_fee_sol = _pt['fr_fee_sol'].sum()
total_br_fee_sol = _pt['br_fee_sol'].sum()
total_fr_txs     = _pt['front_count'].sum()
total_br_txs     = _pt['back_count'].sum()
avg_att_per_tx   = (total_fr_fee_sol + total_br_fee_sol) / (total_fr_txs + total_br_txs)
avg_fr_per_tx    = total_fr_fee_sol / total_fr_txs
avg_br_per_tx    = total_br_fee_sol / total_br_txs

print(f'\n{SEP}')
print('  PER-TRANSACTION FEE AVERAGES  (SOL)')
print(SEP)
print(f'  {"Avg fee per attacker tx (fr+br)":<35} {avg_att_per_tx:.6f}')
print(f'  {"Avg fee per fr tx":<35} {avg_fr_per_tx:.6f}')
print(f'  {"Avg fee per br tx":<35} {avg_br_per_tx:.6f}')
print(f'  (total fr txs: {total_fr_txs:,}  br txs: {total_br_txs:,})')
print(SEP)


In [ ]:
# ── Cell Fee-3d: Per-tx cost — data prep + all-sandwiches box plot ────────
# Victim tip: query inBundle victim txs → resolve bundleId → get tip
# fr_cost_per_tx = fr_fee/front_count  + att_tip/(front_count+back_count)
# br_cost_per_tx = br_fee/back_count   + att_tip/(front_count+back_count)
# vic_cost_per_tx= victim_fee/victim_count + vic_tip/victim_count

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# ── 1. Query victim bundle tips ───────────────────────────────────────────
sw_sids_str = "','".join(all_sw['sandwichId'].tolist())
print('Querying inBundle victim txs...')
vic_sig_df = client.query_df(
    f"""
    SELECT st.sandwichId, st.signature, st.slot
    FROM sandwich_txs st
    WHERE st.sandwichId IN ('{sw_sids_str}')
      AND st.type = 'victim'
      AND st.inBundle = 1
    """,
    settings={'max_query_size': 10_000_000}
)
print(f'  {len(vic_sig_df):,} inBundle victim txs across {vic_sig_df["sandwichId"].nunique():,} sandwiches')

vic_tip_df = None
if len(vic_sig_df) > 0:
    slots_str = ','.join(map(str, vic_sig_df['slot'].unique()))
    jb_vic = client.query_df(
        f"""SELECT bundleId, arrayJoin(transactions) AS sig, landedTipLamports AS tip_lamports
        FROM jito_bundles WHERE slot IN ({slots_str})"""
    )
    vic_sig_df = vic_sig_df.merge(jb_vic, left_on='signature', right_on='sig', how='left')
    vic_tip_df = (
        vic_sig_df.dropna(subset=['bundleId'])
        .drop_duplicates(subset=['sandwichId', 'bundleId'])
        .groupby('sandwichId', as_index=False)['tip_lamports'].sum()
        .rename(columns={'tip_lamports': 'vic_tip_lamports'})
    )
    print(f'  {len(vic_tip_df):,} sandwiches with victim bundle tip')

# ── 2. Build cost dataframe ───────────────────────────────────────────────
_e = (
    fee_sums[['sandwichId', 'fr_fee', 'victim_fee', 'br_fee']]
    .merge(all_sw[['sandwichId', 'sw_class',
                   'front_count', 'victim_count', 'back_count']],
           on='sandwichId')
    .merge(sw_cost[['sandwichId', 'tip_lamports']], on='sandwichId', how='left')
)
_e['tip_lamports'] = _e['tip_lamports'].fillna(0)
if vic_tip_df is not None:
    _e = _e.merge(vic_tip_df, on='sandwichId', how='left')
else:
    _e['vic_tip_lamports'] = 0
_e['vic_tip_lamports'] = _e['vic_tip_lamports'].fillna(0)

_e['block_type']     = _e['sw_class'].map(lambda x: 'cross-block' if x == 'cross_block' else 'in-block')
_e['att_tx_count']   = _e['front_count'] + _e['back_count']
_e['tip_per_att_tx'] = _e['tip_lamports']     / _e['att_tx_count'].clip(lower=1)
_e['vic_tip_per_tx'] = _e['vic_tip_lamports'] / _e['victim_count'].clip(lower=1)
_e['fr_cost_per_tx']  = _e['fr_fee']     / _e['front_count'].clip(lower=1)  + _e['tip_per_att_tx']
_e['vic_cost_per_tx'] = _e['victim_fee'] / _e['victim_count'].clip(lower=1) + _e['vic_tip_per_tx']
_e['br_cost_per_tx']  = _e['br_fee']     / _e['back_count'].clip(lower=1)   + _e['tip_per_att_tx']
_e['fr_cost_per_tx'] = _e['fr_cost_per_tx'] / 1e9
_e['vic_cost_per_tx'] = _e['vic_cost_per_tx'] / 1e9
_e['br_cost_per_tx']  = _e['br_cost_per_tx'] / 1e9

TX_COLS   = ['fr_cost_per_tx', 'vic_cost_per_tx', 'br_cost_per_tx']
TX_LABELS = ['Frontrun', 'Victim', 'Backrun']
TX_COLORS = ['#4472C4', '#E67E22', '#E05252']

def _boxplot(ax, sub, title):
    data = [sub[col].dropna().values for col in TX_COLS]
    bp = ax.boxplot(
        data, patch_artist=True, widths=0.55, showfliers=False,
        medianprops=dict(color='black', linewidth=2),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
    )
    for patch, color in zip(bp['boxes'], TX_COLORS):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    # ax.set_yscale('log')
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(TX_LABELS, fontsize=25)
    ax.set_title(f'{title}', fontsize=25)
    ax.tick_params(axis='y', labelsize=24)
    ax.set_axisbelow(True)
    ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.4, which='both')

# ── 3. Figure 1: All sandwiches ───────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(7, 6))
_boxplot(ax1, _e, 'All sandwiches')
ax1.set_ylabel('Transaction Cost (SOL)', fontsize=22)

pdf1 = 'sandwich_cost_per_tx_box_all.pdf'
with PdfPages(pdf1) as pdf:
    pdf.savefig(fig1, bbox_inches='tight')
plt.show()
print(f'Saved: {pdf1}')

def _print_stats(sub, label):
    SEP = '=' * 65
    print(f'\n{SEP}')
    print(f'  {label}  (lamports)')
    print(SEP)
    print(f'  {"":18} {"median":>12} {"min":>12} {"max":>12}')
    print(f'  {"-"*58}')
    for col, lbl in zip(TX_COLS, ['Frontrun', 'Victim', 'Backrun']):
        r = sub[col].dropna()
        print(f'  {lbl:<18} {r.median():>12,.0f} {r.min():>12,.0f} {r.max():>12,.0f}')
    print(SEP)

_print_stats(_e, 'All sandwiches')


In [ ]:
# ── Cell Fee-3e: Per-tx cost — in-block | cross-block box plots ──────────
# Requires _e, _boxplot from Cell Fee-3d

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

fig2, axes2 = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
fig2.subplots_adjust(wspace=0.06)

for ax, bt in zip(axes2, ['in-block', 'cross-block']):
    _boxplot(ax, _e[_e['block_type'] == bt], bt.title())

axes2[0].set_ylabel('Transaction Cost (SOL)', fontsize=25)

pdf2 = 'sandwich_cost_per_tx_box_split.pdf'
with PdfPages(pdf2) as pdf:
    pdf.savefig(fig2, bbox_inches='tight')
plt.show()
print(f'Saved: {pdf2}')

for bt in ['in-block', 'cross-block']:
    _print_stats(_e[_e['block_type'] == bt], bt)


In [ ]:
# ── Cell: Load all bot_signers from 3_signer_filter ─────────────────────
import glob

SIGNER_DIR = f'{DATA_DIR}'  # DATA_DIR already points to 3_signer_filter root

dfs = []
for cat in CATEGORIES:
    pattern = f'{SIGNER_DIR}/{cat}/bot_signers_{TAG}.parquet'
    files = glob.glob(pattern)
    if not files:
        print(f'  [WARN] no file found: {pattern}')
        continue
    df = pd.read_parquet(files[0])
    df['category'] = cat
    df.index.name = 'signer'
    dfs.append(df.reset_index())

bot_signers = pd.concat(dfs, ignore_index=True)
# signer may appear in multiple categories — keep all rows
print(f'Total rows: {len(bot_signers):,}  |  unique signers: {bot_signers["signer"].nunique():,}')
print(f'category dist:\n{bot_signers["category"].value_counts().to_string()}')
show(bot_signers.sort_values(by='usd_total_profit', ascending=False).head(20))


In [ ]:
# ── Known program names ───────────────────────────────────────────────────
PROGRAM_NAMES = {
    # builtin
    'ComputeBudget111111111111111111111111111111': 'ComputeBudget',
    'ATokenGPvbdGVxr1b2hvZbsiqW5xWH25efTNsLJA8knL': 'Associated Token Account Program',
    '11111111111111111111111111111111': 'System Program',
    'TokenkegQfeZyiNwAJbNbGKPFXCWuBvf9Ss623VQ5DA': 'Token Program',
    'TokenzQdBNbLqP5VEhdkAS6EPFLC1PHnBqCXEpPxuEb': 'Token 2022 Program',
    # DEX / AMM
    'pAMMBay6oceH9fJKBRHGP5D4bD4sWpmSwMn52FMfXEA': 'Pump.fun AMM',
    '6EF8rrecthR5Dkzon8Nwu78hRvfCKubJ14M5uBEwF6P': 'Pump.fun',
    '675kPX9MHTjS2zt1qfr1NYHuzeLXfQM9H24wFSUt1Mp8': 'Raydium Liquidity Pool V4',
    'routeUGWgWzqBWFcrCfv8tritsqukccJPu3q5GPP3xS': 'Raydium AMM Routing',
    'CPMMoo8L3F4NbTegBCKVNunggL7H1ZpdTHKxQB5qKP1C': 'Raydium CPMM',
    'CAMMCzo5YL8w4VFF8KVHrK22GGUsp5VTaW7grrKgrWqK': 'Raydium Concentrated Liquidity',
    'Eo7WjKq67rjJQSZxS6z3YkapzY3eMj6Xy8X5EQVn5UaB': 'Meteora Pools Program',
    'LBUZKhRxPF3XUpBCjp4YzTKgLccjZhTSDM9YuVaPwxo': 'Meteora DLMM Program',
    'dbcij3LWUppWqq96dh6gJWwBifmcGfLSB5D4DuSMaqN': 'Meteora Dynamic Bonding Curve',
    'cpamdpZCGKUy5JxQXB4dcpGPiikHawvSWAd6mEn1sGG': 'Meteora DAMM v2',
    '9W959DqEETiGZocYWCQPaJ6sBmUzgfxXfqGeTEdp3aQP': 'Orca Token Swap V2',
    'DjVE6JNiYqPL2QXyCUUh8rNjHrbz9hXHNYt99MQ59qw1': 'Orca Token Swap V1',
    'whirLbMiicVdio4qvUfM5KAg6Ct8VwpYzGff3uctyCc': 'Whirlpools Program',
    'HpNfyc2Saw7RKkQd8nEL4khUcuPhQ7WwY1B2qjx8jxFq': 'PancakeSwap',
    'Dooar9JkhdZ7J3LHN3A7YCuoGRUggXhQaG4kijfLGU2j': 'StepN DOOAR Swap',
    'REALQqNEomY6cQGZJUGwywTBD2UmDT32rZcNnfxQ5N2': 'Byreal: CLMM',
    # PropAMM
    'SV2EYYJyRz2YhfXwXnhNAevDEui5Q6yrfyo13WtupPF': 'SolFi V2',
    'BiSoNHVpsVZW2F7rx2eQ59yQwKxzU5NvBcmKshCSUypi': 'BisonFi',
    '9H6tua7jkLhdm3w8BvgpTn5LZNU7g4ZynDmCiNN3q6Rp': 'HumidiFi',
    'fUSioN9YKKSa3CUC2YUc4tPkHJ5Y6XW1yz8y6F7qWz9': 'Fusion AMM',
    'TessVdML9pBGgG9yGks7o4HewRaXVAMuoVj4x83GLQH': 'Tessera V',
    'goonuddtQRrWqqn5nFyczVKaie28f3kDkHWkHtURSLE': 'GoonFi V2',
    'ALPHAQmeA7bjrVuccPsYPiCvsi428SNwte66Srvs4pHA': 'AlphaQ',
    'obriQD1zbpyLz95G5n7nJe6a4DPjpFwa5XYPoNm113y': 'Obric V2',
    'ZERor4xhbUycZ6gb9ntrhqscUcZmAbQDjEAtCf4hbZY': 'ZeroFi',
    # Pools
    'GpMZbSM2GgvTKHJirzeGfMFoaZ8UR2X7F4v8vHTvxFbL': 'Raydium Vault Authority 2',
    '5Q544fKrFoe6tsEbD7S8EmxGTJYAKtTVhAW5Q5pge4j1': 'Raydium Authority V4',
    'HLnpSz9h2S4hiLQ43rnSD9XkcUThA7B8hQMKmDaiTLcC': 'Meteora Pool Authority',
    'FhVo3mqL8PW5pH5U2CN4XE33DokiyZnUwuGpH2hmHLuM': 'Meteora DBC: Pool Authority',
    'GF8SKKobum6UJnhX2mLHePU38htg5vdr9zcY4jH8Pqs2': 'AlphaQ (USDT-USDC) Pool 1',
    'F2KCaXcp7AoQtxTDvNEDCyMyWjSCAMWNzcyN9dsPfPs5': 'AlphaQ (JupUSD-USDC) Pool 2',
    'SCoRcH8c2dpjvcJD6FiPbCSQyQgu3PcUAWj2Xxx3mqn': 'SCorch',
    # Aggregators
    '6m2CDdhRgxpH4WjvdzxAYbGxwdGUz5MziiL5jek2kBma': 'OKX DEX: Aggregation Router V2',
    'JUP6LkbZbjS1jKKwapdHNy74zcZ3tLUZoi5QNyVTaV4': 'Jupiter Aggregator v6',
    'DCA265Vj8a9CEuX1eb1LWRnDT7uK6q1xMipnNyatn23M': 'Jupiter DCA program',
    'DF1ow4tspfHX9JwWJsAb9epbkA8hmpSEAtxXy1V27QBH': 'DFlow Aggregator v4',
    'AxiomfHaWDemCFBLBayqnEnNwE6b7B2Qz3UmzMpgbMG6': 'Axiom: Trading Program 1',
}

# ── Cell: Program usage in fr/br txs across all_sw sandwiches ────────────
sw_sids_str = "','".join(all_sw['sandwichId'].tolist())

print('Querying distinct (sandwichId, program) pairs for fr/br txs...')
prog_df = client.query_df(
    f"""
    SELECT DISTINCT
        sandwichId,
        arrayJoin(programs) AS program
    FROM sandwich_txs
    WHERE sandwichId IN ('{sw_sids_str}')
      AND type IN ('frontRun', 'backRun')
      AND length(programs) > 0
    """,
    settings={'max_query_size': 10_000_000}
)
print(f'  {len(prog_df):,} (sandwichId, program) pairs  |  '
      f'{prog_df["program"].nunique():,} unique programs')

# Merge with all_sw — cast signer to str to avoid category dtype issues
_meta = all_sw[['sandwichId', 'signer', 'usd_profit']].copy()
_meta['signer'] = _meta['signer'].astype(str)
prog_df = prog_df.merge(_meta, on='sandwichId', how='left')

# Numeric stats
prog_stats = (
    prog_df.groupby('program', as_index=False)
    .agg(
        sandwich_count = ('sandwichId', 'nunique'),
        signer_count   = ('signer',     'nunique'),
        profit_min     = ('usd_profit', 'min'),
        profit_max     = ('usd_profit', 'max'),
        profit_median  = ('usd_profit', 'median'),
        profit_mean    = ('usd_profit', 'mean'),
        profit_sum     = ('usd_profit', 'sum'),
    )
)

# Signer list separately
signer_lists = (
    prog_df.groupby('program')['signer']
    .apply(lambda x: sorted(x.dropna().unique().tolist()))
    .reset_index()
    .rename(columns={'signer': 'signers'})
)
prog_stats = prog_stats.merge(signer_lists, on='program', how='left')
prog_stats['name'] = prog_stats['program'].map(PROGRAM_NAMES).fillna('')
all_profit = all_sw['usd_profit'].sum()
prog_stats['profit_share'] = prog_stats['profit_sum'] / all_profit * 100
prog_stats['volume_share'] = prog_stats['sandwich_count'] / len(all_sw) * 100
print(f'Total unique programs: {len(prog_stats):,}')

DISPLAY_COLS = ['name', 'program', 'signer_count', 'sandwich_count', 'volume_share', 'profit_sum', 'profit_share',
                'profit_min', 'profit_max', 'profit_median',
                'profit_mean']

# ── Table 1: top 50 by sandwich count ─────────────────────────────────────
top_by_count = (prog_stats.sort_values('sandwich_count', ascending=False)
                           .head(50).reset_index(drop=True))
print('\n' + '='*70)
print('  TOP 50 PROGRAMS  by sandwich count')
print('='*70)
display(top_by_count[DISPLAY_COLS])

# ── Table 2: top 50 by total profit ───────────────────────────────────────
top_by_profit = (prog_stats.sort_values('profit_sum', ascending=False)
                            .head(50).reset_index(drop=True))
print('\n' + '='*70)
print('  TOP 50 PROGRAMS  by total profit (USD)')
print('='*70)
display(top_by_profit[DISPLAY_COLS])


In [ ]:
# ── Usage of two specific programs ──────────────────────────────────────
TARGET_PROGRAMS = {
    'pAMMBay6oceH9fJKBRHGP5D4bD4sWpmSwMn52FMfXEA': 'pAMM',
    '6EF8rrecthR5Dkzon8Nwu78hRvfCKubJ14M5uBEwF6P': 'Pump.fun',
}

SEP = '=' * 60
print(SEP)
print('  ATTACKER USAGE OF SPECIFIC PROGRAMS')
print(SEP)

rows = []
for prog, name in TARGET_PROGRAMS.items():
    # sandwiches that used this program
    sw_using = prog_df[prog_df['program'] == prog]['sandwichId'].unique()
    # signers behind those sandwiches
    signers = all_sw[all_sw['sandwichId'].isin(sw_using)]['signer'].nunique()
    sw_count = len(sw_using)
    rows.append({
        'Program': f'{name} ({prog[:8]}...)',
        'Sandwiches using it': sw_count,
        'Unique signers': signers,
    })
    print(f'\n  {name} ({prog[:8]}...)')
    print(f'    Sandwiches using it: {sw_count:>8,}')
    print(f'    Unique signers:      {signers:>8,}')

print(f'\n{SEP}')
display(pd.DataFrame(rows))

# Combined: signers who used either program
all_sw_using = prog_df[prog_df['program'].isin(TARGET_PROGRAMS)]['sandwichId'].unique()
combined_signers = all_sw[all_sw['sandwichId'].isin(all_sw_using)]['signer'].nunique()
print(f'\n  Combined unique signers (either program): {combined_signers:,}')


In [ ]:
import matplotlib.cm as cm

# ── Cell: Stacked-area — sandwich volume & profit share by program ────────
# ── CONFIG ────────────────────────────────────────────────────────────────
SELECTED_PROGRAMS = [
    'Bgo4vNe3vxRv37j8mmQarJo8vbjHEKgkhJDZzxiizBid',
    '4WYNb3xxJRVUadXRDE1q3FyzMdQ2zzzEijdR1S7pGLMu',
    'EpjnUueXY3MmZZw2n8FobumsJpZN6kTUB8mREr2SKXRW',
    'EumFF6mx6mTPW9VSJsPmrmqgLF9amvzKwXfQfyrkwMTH',
    '6y45KerAR8S4nfx7CDTpY8XWqqffduywhsHFLVUojh5U',
    '8jXneHCUpbGQpSmSz9TgJhhXCW2vJyzbtXZLT9xrLyon',
    'HawkGoDqx9gTnU3MPWh5HLygM3hn5LRLvnKUxzPvtjwM',
    'GxkuFR1WtkniWPT5xpf4FVqgD929WgFHyy8F82oMuoce',
    '69f3WXVx1JXRTZ5aELcjvi46GWGvd6iHmkoyAPiPQyoU',
    '4bwoPeP9UTEF7evUR6jkLZP4RbEJr2mULfxiMTa11mQG',
    # '9FcKTaqDQHfhuRvqjtmUpMiMJtVdbQWa6M8WuDThqqbx'
]
Y_UPPER_VOL    = 0.8
Y_UPPER_PROFIT = 0.8

PROGRAM_COLORS = [
    '#05678d',  # 0  dark navy
    '#03c39b',  # 1  teal green
    '#f8bf57',  # 2  golden yellow
    '#da3e52',  # 3  brick red
    '#3c096c',  # 4  deep purple
    '#f3bac9',  # 5  light pink
    '#02afd5',  # 6  steel blue
    '#7e6b8f',  # 8  orange
    '#c1cc99',  # 7  sage green
    '#b217c8',  # 9  violet
]
REST_COLOR = '#d9f7fb'  # light blue — rest (topmost/background)
# ── END CONFIG ────────────────────────────────────────────────────────────

import matplotlib.ticker as mticker

assert SELECTED_PROGRAMS, 'Fill in SELECTED_PROGRAMS first'

sig_counts = prog_stats.set_index('program')['signer_count'].to_dict()

def prog_label(p):
    n = sig_counts.get(p, '?')
    return f'{p[:5]} ({n})'

# color_map = {p: PROGRAM_COLORS[i] for i, p in enumerate(SELECTED_PROGRAMS)}
# color_map['rest'] = REST_COLOR

cmap = cm.get_cmap('tab20c')  # or 'tab20', 'Set3', 'viridis'
color_map = {
    p: cmap(i / max(len(SELECTED_PROGRAMS) - 1, 1))
    for i, p in enumerate(SELECTED_PROGRAMS)
}
color_map['rest'] = (0.95, 0.95, 0.95, 1.0)

priority = {p: i for i, p in enumerate(SELECTED_PROGRAMS)}

sw_prog = (
    prog_df[prog_df['program'].isin(set(SELECTED_PROGRAMS))]
    .assign(_pri=lambda d: d['program'].map(priority))
    .sort_values('_pri')
    .drop_duplicates(subset=['sandwichId'], keep='first')
    [['sandwichId', 'program']]
)

matched = set(sw_prog['sandwichId'])
rest_df = pd.DataFrame({
    'sandwichId': [s for s in all_sw['sandwichId'] if s not in matched],
    'program': 'rest',
})
sw_full = pd.concat([sw_prog, rest_df], ignore_index=True)

sw_full = sw_full.merge(
    all_sw[['sandwichId', 'ts', 'usd_profit']], on='sandwichId', how='left'
)
sw_full['date'] = pd.to_datetime(sw_full['ts']).dt.floor('D')

daily_total = sw_full.groupby('date').agg(
    total_count  = ('sandwichId', 'nunique'),
    total_profit = ('usd_profit', 'sum'),
).reset_index()

daily_prog = sw_full.groupby(['date', 'program']).agg(
    count  = ('sandwichId', 'nunique'),
    profit = ('usd_profit', 'sum'),
).reset_index()

daily_prog = daily_prog.merge(daily_total, on='date')
daily_prog['vol_share']    = daily_prog['count']  / daily_prog['total_count']
daily_prog['profit_share'] = daily_prog['profit'] / daily_prog['total_profit']

all_progs     = SELECTED_PROGRAMS + ['rest']
global_vol    = (daily_prog.groupby('program')['count'].sum()
                 .reindex(SELECTED_PROGRAMS, fill_value=0))
global_profit = (daily_prog.groupby('program')['profit'].sum()
                 .reindex(SELECTED_PROGRAMS, fill_value=0))

# rest always on top (last); named programs largest-at-bottom
vol_order    = global_vol.sort_values(ascending=False).index.tolist() + ['rest']
profit_order = global_profit.sort_values(ascending=False).index.tolist() + ['rest']

vol_pivot    = (daily_prog.pivot(index='date', columns='program', values='vol_share')
                          .reindex(columns=all_progs, fill_value=0).fillna(0))
profit_pivot = (daily_prog.pivot(index='date', columns='program', values='profit_share')
                          .reindex(columns=all_progs, fill_value=0).fillna(0))

dates = vol_pivot.index

# ── Plot ──────────────────────────────────────────────────────────────────
def _make_share_fig(pivot, order, y_upper, ylabel, pdf_name):
    ordered_pivot  = pivot[order]
    colors_ordered = [color_map[p] for p in order]
    labels_ordered = [prog_label(p) if p != 'rest' else 'rest' for p in order]

    fig, ax = plt.subplots(figsize=(20, 12))
    fig.subplots_adjust(top=0.80)

    ax.stackplot(dates, ordered_pivot.T.values,
                 labels=labels_ordered, colors=colors_ordered, alpha=0.88)
    ax.set_xlim(dates[0], dates[-1])
    ax.set_ylim(0, y_upper)
    ax.set_ylabel(ylabel, fontsize=38)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
    ax.xaxis.set_minor_locator(mdates.DayLocator(interval=1))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=9)
    ax.tick_params(axis='x', which='minor', length=3, color='#cccccc')
    ax.tick_params(axis='x', which='major', length=6, labelsize=30)
    ax.tick_params(axis='y', labelsize=30)
    ax.set_axisbelow(True)
    ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.4, color='#aaaaaa')

    handles, lbls = ax.get_legend_handles_labels()
    fig.legend(handles, lbls,
               title='Program (# attackers)', title_fontsize=30,
               fontsize=30, framealpha=0.9,
               loc='upper center', bbox_to_anchor=(0.5, 1.0),
               ncol=6, handlelength=1.2, columnspacing=0.8)

    with PdfPages(pdf_name) as pdf:
        pdf.savefig(fig, bbox_inches='tight')
    plt.show()
    print(f'Saved: {pdf_name}')

_make_share_fig(vol_pivot,    vol_order,    Y_UPPER_VOL,    'Sandwich Volume Share (%)',
                'sandwich_program_stack_vol.pdf')
_make_share_fig(profit_pivot, profit_order, Y_UPPER_PROFIT, 'Daily Sandwich Profit Share (%)',
                'sandwich_program_stack_profit.pdf')

# ── Side-by-side combined figure ──────────────────────────────────────────
def _make_share_fig_side_by_side(
    vol_pivot, vol_order, profit_pivot, profit_order,
    y_upper_vol, y_upper_profit, pdf_name
):
    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    fig.subplots_adjust(top=0.78, wspace=0.12)

    panels = [
        (axes[0], vol_pivot,    vol_order,    y_upper_vol,    'Daily sandwich volume share'),
        (axes[1], profit_pivot, profit_order, y_upper_profit, 'Daily sandwich profit share'),
    ]
    for ax, pivot, order, y_upper, ylabel in panels:
        ordered_pivot  = pivot[order]
        colors_ordered = [color_map[p] for p in order]
        labels_ordered = [prog_label(p) if p != 'rest' else 'rest' for p in order]

        ax.stackplot(dates, ordered_pivot.T.values,
                     labels=labels_ordered, colors=colors_ordered, alpha=0.88)
        ax.set_xlim(dates[0], dates[-1])
        ax.set_ylim(0, y_upper)
        ax.set_ylabel(ylabel, fontsize=28)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
        ax.xaxis.set_minor_locator(mdates.DayLocator(interval=1))
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=9)
        ax.tick_params(axis='x', which='minor', length=3, color='#cccccc')
        ax.tick_params(axis='x', which='major', length=5, labelsize=10)
        ax.tick_params(axis='y', labelsize=10)
        ax.set_axisbelow(True)
        ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.4, color='#aaaaaa')

    # Shared legend from left panel
    handles, lbls = axes[0].get_legend_handles_labels()
    fig.legend(handles, lbls,
               title='Program (# attackers)', title_fontsize=12,
               fontsize=10, framealpha=0.9,
               loc='upper center', bbox_to_anchor=(0.5, 1.0),
               ncol=6, handlelength=1.2, columnspacing=0.8)

    with PdfPages(pdf_name) as pdf:
        pdf.savefig(fig, bbox_inches='tight')
    plt.show()
    print(f'Saved: {pdf_name}')

_make_share_fig_side_by_side(
    vol_pivot, vol_order, profit_pivot, profit_order,
    Y_UPPER_VOL, Y_UPPER_PROFIT,
    'sandwich_program_stack_combined.pdf'
)


In [ ]:
# ── Cell Fee-Summary: global cost & tip stats ───────────────────────────
# Requires: sw_cost, tip_df2, all_sw  (Fee-1 / Fee-2 / Fee-3 must have run)

SOL = 1e9

total_att_fee_sol  = (sw_cost['fr_fee'] + sw_cost['br_fee']).sum() / SOL
total_vic_fee_sol  = sw_cost['victim_fee'].sum() / SOL
total_all_fee_sol  = (sw_cost['fr_fee'] + sw_cost['victim_fee'] + sw_cost['br_fee']).sum() / SOL
total_tip_sol      = tip_df2['tip_lamports'].sum() / SOL
mean_att_fee_sol   = (sw_cost['fr_fee'] + sw_cost['br_fee']).mean() / SOL
mean_tip_sol       = tip_df2['tip_lamports'].mean() / SOL
n_bundle           = len(tip_df2)
n_total            = len(sw_cost)

SEP = '=' * 57
print(SEP)
print('  GLOBAL FEE & TIP SUMMARY')
print(SEP)
print(f'  {"Metric":<40} {"SOL":>12}')
print(f'  {"-"*54}')
print(f'  {"Total attacker fee (fr+br)":<40} {total_att_fee_sol:>12.2f}')
print(f'  {"Total victim fee":<40} {total_vic_fee_sol:>12.2f}')
print(f'  {"Total all fees (fr+vic+br)":<40} {total_all_fee_sol:>12.2f}')
print(f'  {"Total tips (bundle sandwiches)":<40} {total_tip_sol:>12.2f}')
print(SEP)
print(f'  {"Avg attacker fee per sandwich (fr+br)":<40} {mean_att_fee_sol:>12.6f}')
print(f'  {"Avg tip per bundle sandwich":<40} {mean_tip_sol:>12.6f}')
print(f'    (bundle sandwiches with tip: {n_bundle:,} / total: {n_total:,},'
      f' {100*n_bundle/n_total:.1f}%)')
print(SEP)
